In [ ]:
# Cell 0: Setup — install deps + write all source files
import subprocess, os, torch, base64

subprocess.run(['pip', 'install', '-q', 'torch',
                '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run(['pip', 'install', '-q', 'numpy', 'scipy', 'pandas',
                'scikit-learn', 'tqdm', 'pytorch_metric_learning'], check=True)

REPO = '/kaggle/working/motion-id'
FILES = {"configs/__init__.py": "IyBjb25maWdzIHBhY2thZ2UK", "configs/config.py": "ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgTGlzdAoKCkBkYXRhY2xhc3MKY2xhc3MgQ29uZmlnOgogICAgIyDilIDilIAgTVBJIChwYXBlciBTZWN0aW9uIDQuMS4xICsgNC4zLjEpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgbXBpX3NhbXBsaW5nX3JhdGU6IGludCA9IDUwCiAgICBtcGlfd2luZG93X3NlYzogZmxvYXQgPSAzLjAgICAgICAgICAgIyBwYXBlcjogImdhdGhlcmVkIGRhdGEgb3ZlciAzIHNlY29uZHMiCiAgICBtcGlfbWluX3JlYWRpbmdzOiBpbnQgPSAxMDAgICAgICAgICAgIyBwYXBlcjogImF0IGxlYXN0IDEwMCByZWFkaW5ncyBmb3IgZWFjaCBzZW5zb3IiCiAgICBtcGlfbl9jaGFubmVsczogaW50ID0gMTggICAgICAgICAgICAjIDYgc2Vuc29ycyDDlyAzIGF4ZXMKICAgIG1waV9lcG9jaHM6IGludCA9IDMwCiAgICBzdGF0aW9uYXJ5X3RocmVzaG9sZDogZmxvYXQgPSAwLjAxICAjIG5lYXItemVybyBsaW5fYWNjIChtL3PCsikKCiAgICAjIOKUgOKUgCBVViAocGFwZXIgU2VjdGlvbiA0LjEuMiArIDQuMy4yKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHV2X3NhbXBsaW5nX3JhdGU6IGludCA9IDUwCiAgICB1dl93aW5kb3dfc2VjOiBmbG9hdCA9IDEuMCAgICAgICAgICAgIyBwYXBlcjogIm9ubHkgaW50ZXJlc3RlZCBpbiBsYXN0IHNlY29uZCIKICAgIHV2X2F1Z21lbnRfd2luZG93X3NlYzogZmxvYXQgPSAxLjUgICMgcGFwZXI6ICIxLjUgc2Vjb25kcyByYW5kb21seSBjdXQgdG8gMXMiCiAgICB1dl9uX2ZlYXR1cmVzOiBpbnQgPSAyMiAgICAgICAgICAgICAjIHBhcGVyOiAidG90YWwgbnVtYmVyIG9mIGZlYXR1cmUgdmVjdG9ycyB3YXMgMjIiCiAgICB1dl9uX2NoYW5uZWxzX3Blcl9mZWF0dXJlOiBpbnQgPSA0ICAjIDMtY2ggZmVhdHVyZXMgemVyby1wYWRkZWQgdG8gNAogICAgdXZfbl90cmlhbHM6IGludCA9IDMwMCAgICAgICAgICAgICAgIyBwYXBlcjogMzAwIGxpZnRzIHBlciB1c2VyCiAgICB1dl9uX2NsdXN0ZXJzOiBpbnQgPSA2ICAgICAgICAgICAgICMgcGFwZXI6IDYgbG9jYXRpb25zCiAgICB1dl90b3RhbF91c2VyczogaW50ID0gMTAxCiAgICB1dl90ZXN0X3VzZXJzOiBpbnQgPSAxMQoKICAgICMg4pSA4pSAIFRyYWluaW5nIChwYXBlciBTZWN0aW9uIDYuNSkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICB1dl9iYXNlbGluZV9uOiBpbnQgPSA3NQogICAgdXZfbl9zcGxpdHM6IExpc3RbaW50XSA9IGZpZWxkKAogICAgICAgIGRlZmF1bHRfZmFjdG9yeT1sYW1iZGE6IFs2MCwgNjUsIDcwLCA3NSwgODAsIDg1XSkKICAgIGJhc2VsaW5lX2xyOiBmbG9hdCA9IDFlLTMgICAgICAgICAgICMgbm90IGluIHBhcGVyCiAgICBmaW5ldHVuZV9scjogZmxvYXQgPSAxZS00ICAgICAgICAgICAjIG5vdCBpbiBwYXBlcgogICAgYmFzZWxpbmVfZXBvY2hzOiBpbnQgPSA1MCAgICAgICAgICAgIyBub3QgaW4gcGFwZXIKICAgIGZpbmV0dW5lX2Vwb2NoczogaW50ID0gMTAgICAgICAgICAgICMgbm90IGluIHBhcGVyCiAgICBiYXRjaF9zaXplOiBpbnQgPSA2NCAgICAgICAgICAgICAgICAjIG5vdCBpbiBwYXBlcgogICAgYWxwaGFfdG06IGZsb2F0ID0gMS4wICAgICAgICAgICAgICAgIyBub3QgaW4gcGFwZXIKICAgIHN1cGNvbl90ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjA3ICAgICMgS2hvc2xhIGV0IGFsLiAyMDIwIGRlZmF1bHQKICAgIG5fdHJhaW5pbmdfcnVuczogaW50ID0gNSAgICAgICAgICAgICMgcGFwZXIgdHJhaW5zIDUgdGltZXMsIHJlcG9ydHMgbWVhbsKxc3RkCgogICAgIyDilIDilIAgRXZhbHVhdGlvbiAocGFwZXIgU2VjdGlvbiA0LjIgKyA3KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHRhcl90aHJlc2hvbGQ6IGZsb2F0ID0gMC45MCAgICAgICAgICMgcGFwZXI6IFRBUihARkFSPTEvNTAwMDApPTkwJQogICAgdGFyZ2V0X2ZhcjogZmxvYXQgPSAxIC8gNTAwMDAgICAgICAgIyBwYXBlcjogQW5kcm9pZCBDREQgQ2xhc3MgMwogICAgYm9vdHN0cmFwX3JlcGVhdHM6IGludCA9IDUwMDAgICAgICAgIyBwYXBlcjogNTAwMCByZXBlYXRzCiAgICBmYXJfc3dlZXBfc3RlcHM6IGludCA9IDEwMF8wMDAKCgpjZmcgPSBDb25maWcoKQo=", "utils/__init__.py": "IyB1dGlscyBwYWNrYWdlCg==", "utils/bin_reader.py": "IiIiClJlYWRzIE1vdGlvbklEIHNlbnNvciBmaWxlcy4KClRoZSBLYWdnbGUgZGF0YXNldCBzdG9yZXMgc2Vuc29ycyBhcyBmaWxlcyB3aXRoIC50eHQgZXh0ZW5zaW9uIGJ1dCBCSU5BUlkgY29udGVudDoKICBhY2NlbC50eHQsIGdyYXZpdHkudHh0LCBneXJvLnR4dCwgbGluQWNjZWwudHh0LCBNYWduZXRpY0ZpZWxkLnR4dCwgUm90YXRpb24udHh0CiAgRm9ybWF0OiAyMCBieXRlcyBwZXIgcmVjb3JkCiAgICBieXRlcyAwLTc6ICB0aW1lc3RhbXAgbXMsIGJpZy1lbmRpYW4gaW50NjQgICAoc3RydWN0ICc+cScpCiAgICBieXRlcyA4LTE5OiBYIFkgWiB2YWx1ZXMsIGxpdHRsZS1lbmRpYW4gZmxvYXQzMiAoc3RydWN0ICc8ZmZmJykKCnNjcmVlbi50eHQgaXMgYWN0dWFsIHRleHQ6CiAgRm9ybWF0OiAie3RpbWVzdGFtcF9tc30ge2FuZHJvaWQuaW50ZW50LmFjdGlvbi4qfSIKCkVGRklDSUVOVCBMT0FESU5HOiBiaW5hcnkgZmlsZXMgc3VwcG9ydCBzZWVraW5nIHRvIGEgdGltZSBvZmZzZXQsCnNvIHdlIG9ubHkgcmVhZCByZWNvcmRzIHdpdGhpbiB0aGUgbmVlZGVkIHRpbWUgd2luZG93LgoiIiIKCmltcG9ydCBvcwppbXBvcnQgc3RydWN0CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpGTEFHX1VTRVJfUFJFU0VOVCA9ICJhbmRyb2lkLmludGVudC5hY3Rpb24uVVNFUl9QUkVTRU5UIgpGTEFHX1NDUkVFTl9PRkYgICA9ICJhbmRyb2lkLmludGVudC5hY3Rpb24uU0NSRUVOX09GRiIKRkxBR19TQ1JFRU5fT04gICAgPSAiYW5kcm9pZC5pbnRlbnQuYWN0aW9uLlNDUkVFTl9PTiIKClJFQ09SRF9TSVpFID0gMjAgICMgYnl0ZXMgcGVyIHNlbnNvciByZWNvcmQKClNFTlNPUl9GSUxFUyA9IHsKICAgICJhY2MiOiAgImFjY2VsLnR4dCIsCiAgICAiZ3JhdiI6ICJncmF2aXR5LnR4dCIsCiAgICAiZ3lybyI6ICJneXJvLnR4dCIsCiAgICAibGluIjogICJsaW5BY2NlbC50eHQiLAogICAgIm1hZyI6ICAiTWFnbmV0aWNGaWVsZC50eHQiLAogICAgInJvdCI6ICAiUm90YXRpb24udHh0IiwKfQpTQ1JFRU5fRklMRSA9ICJzY3JlZW4udHh0IgoKU0VOU09SX0NPTFMgPSB7CiAgICBuYW1lOiBbZiJ7bmFtZX1fWCIsIGYie25hbWV9X1kiLCBmIntuYW1lfV9aIl0KICAgIGZvciBuYW1lIGluIFNFTlNPUl9GSUxFUwp9CgpBTExfU0VOU09SX0NPTFMgPSAoCiAgICBTRU5TT1JfQ09MU1siYWNjIl0gICsgU0VOU09SX0NPTFNbImdyYXYiXSArIFNFTlNPUl9DT0xTWyJneXJvIl0gKwogICAgU0VOU09SX0NPTFNbImxpbiJdICArIFNFTlNPUl9DT0xTWyJtYWciXSAgKyBTRU5TT1JfQ09MU1sicm90Il0KKQoKCmRlZiByZWFkX3NlbnNvcl9iaW4ocGF0aDogc3RyLAogICAgICAgICAgICAgICAgICAgIHRfc3RhcnQ6IGludCA9IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgdF9lbmQ6ICAgaW50ID0gTm9uZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiCiAgICBSZWFkIGEgYmluYXJ5IHNlbnNvciBmaWxlICgyMCBieXRlcy9yZWNvcmQpLgogICAgSWYgdF9zdGFydC90X2VuZCBnaXZlbiwgdXNlcyBiaW5hcnkgc2VhcmNoIHRvIHNraXAgdG8gdGhlIHJpZ2h0IG9mZnNldC4KICAgIEZpbGUgbXVzdCBiZSBzb3J0ZWQgYnkgdGltZXN0YW1wIChpdCBpcykuCiAgICAiIiIKICAgIGZpbGVfc2l6ZSA9IG9zLnBhdGguZ2V0c2l6ZShwYXRoKQogICAgaWYgZmlsZV9zaXplID09IDAgb3IgZmlsZV9zaXplICUgUkVDT1JEX1NJWkUgIT0gMDoKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKGNvbHVtbnM9WyJ0aW1lc3RhbXAiLCAiWCIsICJZIiwgIloiXSkKCiAgICBuX3JlY29yZHMgPSBmaWxlX3NpemUgLy8gUkVDT1JEX1NJWkUKCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICAjIElmIG5vIHRpbWUgZmlsdGVyLCByZWFkIGV2ZXJ5dGhpbmcKICAgICAgICBpZiB0X3N0YXJ0IGlzIE5vbmUgYW5kIHRfZW5kIGlzIE5vbmU6CiAgICAgICAgICAgIGRhdGEgPSBmLnJlYWQoKQogICAgICAgICAgICByZXR1cm4gX3BhcnNlX2JpbmFyeV9ibG9jayhkYXRhKQoKICAgICAgICAjIEJpbmFyeSBzZWFyY2ggZm9yIHN0YXJ0IG9mZnNldAogICAgICAgIHN0YXJ0X2lkeCA9IF9iaW5hcnlfc2VhcmNoX3RzKGYsIG5fcmVjb3JkcywgdF9zdGFydCBvciAwKQogICAgICAgICMgUmVhZCBmcm9tIHN0YXJ0X2lkeCB0byBlbmQgKG9yIHVudGlsIHRfZW5kKQogICAgICAgIGYuc2VlayhzdGFydF9pZHggKiBSRUNPUkRfU0laRSkKICAgICAgICAjIFJlYWQgYSBjaHVuaywgdGhlbiBmaWx0ZXIgYnkgdGltZXN0YW1wCiAgICAgICAgcmVtYWluaW5nID0gKG5fcmVjb3JkcyAtIHN0YXJ0X2lkeCkgKiBSRUNPUkRfU0laRQogICAgICAgIGNodW5rX3NpemUgPSBtaW4ocmVtYWluaW5nLCAxMF8wMDBfMDAwKSAgIyAxME1CIGNodW5rcwogICAgICAgIGRhdGEgPSBmLnJlYWQoY2h1bmtfc2l6ZSkKICAgICAgICBkZiA9IF9wYXJzZV9iaW5hcnlfYmxvY2soZGF0YSkKICAgICAgICAjIEZpbHRlciBieSB0aW1lIHJhbmdlCiAgICAgICAgaWYgdF9zdGFydCBpcyBub3QgTm9uZToKICAgICAgICAgICAgZGYgPSBkZltkZlsidGltZXN0YW1wIl0gPj0gdF9zdGFydF0KICAgICAgICBpZiB0X2VuZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgZGYgPSBkZltkZlsidGltZXN0YW1wIl0gPD0gdF9lbmRdCiAgICAgICAgcmV0dXJuIGRmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgX3BhcnNlX2JpbmFyeV9ibG9jayhkYXRhOiBieXRlcykgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUGFyc2UgcmF3IGJ5dGVzIGludG8gRGF0YUZyYW1lIHVzaW5nIG51bXB5IChmYXN0KS4iIiIKICAgIG4gPSBsZW4oZGF0YSkgLy8gUkVDT1JEX1NJWkUKICAgIGlmIG4gPT0gMDoKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKGNvbHVtbnM9WyJ0aW1lc3RhbXAiLCAiWCIsICJZIiwgIloiXSkKICAgIGFyciA9IG5wLmZyb21idWZmZXIoZGF0YSwgZHR5cGU9bnAudWludDgpLnJlc2hhcGUobiwgMjApCiAgICAjIFRpbWVzdGFtcDogYnl0ZXMgMC03LCBiaWctZW5kaWFuIGludDY0CiAgICB0cyA9IG5wLnplcm9zKG4sIGR0eXBlPW5wLmludDY0KQogICAgZm9yIGkgaW4gcmFuZ2UoOCk6CiAgICAgICAgdHMgPSB0cyB8IChhcnJbOiwgaV0uYXN0eXBlKG5wLmludDY0KSA8PCAoNTYgLSA4ICogaSkpCiAgICAjIFhZWjogYnl0ZXMgOC0xOSwgbGl0dGxlLWVuZGlhbiBmbG9hdDMyCiAgICB4eXogPSBhcnJbOiwgODoyMF0udmlldyhucC5mbG9hdDMyKS5yZXNoYXBlKG4sIDMpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHsidGltZXN0YW1wIjogdHMsICJYIjogeHl6WzosIDBdLAogICAgICAgICAgICAgICAgICAgICAgICAgIlkiOiB4eXpbOiwgMV0sICJaIjogeHl6WzosIDJdfSkKCgpkZWYgX2JpbmFyeV9zZWFyY2hfdHMoZiwgbl9yZWNvcmRzOiBpbnQsIHRhcmdldF90czogaW50KSAtPiBpbnQ6CiAgICAiIiJCaW5hcnkgc2VhcmNoIGZvciB0aGUgZmlyc3QgcmVjb3JkIHdpdGggdGltZXN0YW1wID49IHRhcmdldF90cy4iIiIKICAgIGxvLCBoaSA9IDAsIG5fcmVjb3JkcyAtIDEKICAgIHdoaWxlIGxvIDwgaGk6CiAgICAgICAgbWlkID0gKGxvICsgaGkpIC8vIDIKICAgICAgICBmLnNlZWsobWlkICogUkVDT1JEX1NJWkUpCiAgICAgICAgdHMgPSBzdHJ1Y3QudW5wYWNrKCI+cSIsIGYucmVhZCg4KSlbMF0KICAgICAgICBpZiB0cyA8IHRhcmdldF90czoKICAgICAgICAgICAgbG8gPSBtaWQgKyAxCiAgICAgICAgZWxzZToKICAgICAgICAgICAgaGkgPSBtaWQKICAgIHJldHVybiBsbwoKCmRlZiByZWFkX3NjcmVlbihwYXRoOiBzdHIpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIgogICAgUmVhZCBzY3JlZW4udHh0LiBIYW5kbGVzIGJvdGggZm9ybWF0czoKICAgICAgInRpbWVzdGFtcCBldmVudCIgICAgICAgICAgICAgICgyIGZpZWxkcyDigJQgS2FnZ2xlIGRhdGFzZXQgZm9ybWF0KQogICAgICAiZGF0ZSB0aW1lIGV2ZW50IiAgICAgICAgICAgICAgKDMgZmllbGRzIOKAlCBvbGRlciBmb3JtYXQpCiAgICBSZXR1cm5zIERhdGFGcmFtZSB3aXRoIGNvbHVtbnMgW3RpbWVzdGFtcCwgZXZlbnRdLgogICAgIiIiCiAgICByb3dzID0gW10KICAgIHdpdGggb3BlbihwYXRoLCAiciIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgcGFydHMgPSBsaW5lLnN0cmlwKCkuc3BsaXQoKQogICAgICAgICAgICBpZiBsZW4ocGFydHMpID09IDI6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJ0aW1lc3RhbXAiOiBpbnQocGFydHNbMF0pLCAiZXZlbnQiOiBwYXJ0c1sxXX0pCiAgICAgICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBlbGlmIGxlbihwYXJ0cykgPT0gMzoKICAgICAgICAgICAgICAgIHBhc3MgICMgb2xkZXIgZm9ybWF0LCBza2lwCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpCgoKZGVmIGxvYWRfc2Vzc2lvbihzZXNzaW9uX2Rpcjogc3RyLAogICAgICAgICAgICAgICAgIHdpbmRvd19zZWNfYmVmb3JlOiBmbG9hdCA9IDYwLjApIC0+IHR1cGxlOgogICAgIiIiCiAgICBMb2FkIGFsbCBzZW5zb3IgZmlsZXMgYW5kIHNjcmVlbiBldmVudHMgZm9yIG9uZSBzZXNzaW9uLgogICAgVXNlcyBiaW5hcnkgc2VhcmNoIHRvIG9ubHkgcmVhZCByZWNvcmRzIGluIHRoZSBuZWVkZWQgdGltZSB3aW5kb3cuCgogICAgUmV0dXJucyAobWVyZ2VkX2RmLCBzY3JlZW5fZGYpIG9yIChOb25lLCBOb25lKSBvbiBmYWlsdXJlLgogICAgIiIiCiAgICAjIENoZWNrIGFsbCBmaWxlcyBleGlzdAogICAgZm9yIG5hbWUsIGZuYW1lIGluIFNFTlNPUl9GSUxFUy5pdGVtcygpOgogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oc2Vzc2lvbl9kaXIsIGZuYW1lKSk6CiAgICAgICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICBzY3JlZW5fcGF0aCA9IG9zLnBhdGguam9pbihzZXNzaW9uX2RpciwgU0NSRUVOX0ZJTEUpCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoc2NyZWVuX3BhdGgpOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCgogICAgIyBSZWFkIHNjcmVlbiBldmVudHMgZmlyc3QgKHNtYWxsIHRleHQgZmlsZSkKICAgIHNjcmVlbl9kZiA9IHJlYWRfc2NyZWVuKHNjcmVlbl9wYXRoKQogICAgaWYgbGVuKHNjcmVlbl9kZikgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZSwgTm9uZQoKICAgICMgVGltZSB3aW5kb3cgZnJvbSBzY3JlZW4gZXZlbnRzCiAgICB0X21pbiA9IGludChzY3JlZW5fZGZbInRpbWVzdGFtcCJdLm1pbigpKSAtIGludCh3aW5kb3dfc2VjX2JlZm9yZSAqIDEwMDApCiAgICB0X21heCA9IGludChzY3JlZW5fZGZbInRpbWVzdGFtcCJdLm1heCgpKSArIDUwMDAKCiAgICAjIExvYWQgZWFjaCBzZW5zb3Igd2l0aCB0aW1lIGZpbHRlcmluZyB2aWEgYmluYXJ5IHNlYXJjaAogICAgc2Vuc29yX2RmcyA9IHt9CiAgICBmb3IgbmFtZSwgZm5hbWUgaW4gU0VOU09SX0ZJTEVTLml0ZW1zKCk6CiAgICAgICAgZnBhdGggPSBvcy5wYXRoLmpvaW4oc2Vzc2lvbl9kaXIsIGZuYW1lKQogICAgICAgIGRmICAgID0gcmVhZF9zZW5zb3JfYmluKGZwYXRoLCB0X3N0YXJ0PXRfbWluLCB0X2VuZD10X21heCkKICAgICAgICBpZiBsZW4oZGYpID09IDA6CiAgICAgICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICAgICAgY29scyA9IFNFTlNPUl9DT0xTW25hbWVdCiAgICAgICAgZGYgICA9IGRmLnJlbmFtZShjb2x1bW5zPXsiWCI6IGNvbHNbMF0sICJZIjogY29sc1sxXSwgIloiOiBjb2xzWzJdfSkKICAgICAgICBzZW5zb3JfZGZzW25hbWVdID0gZGYuc29ydF92YWx1ZXMoInRpbWVzdGFtcCIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCiAgICAjIE1lcmdlIGFsbCBzZW5zb3JzIG9uIHRpbWVzdGFtcAogICAgbWVyZ2VkID0gc2Vuc29yX2Rmc1siYWNjIl0KICAgIGZvciBuYW1lIGluIFsiZ3JhdiIsICJneXJvIiwgImxpbiIsICJtYWciLCAicm90Il06CiAgICAgICAgbWVyZ2VkID0gcGQubWVyZ2VfYXNvZigKICAgICAgICAgICAgbWVyZ2VkLCBzZW5zb3JfZGZzW25hbWVdLAogICAgICAgICAgICBvbj0idGltZXN0YW1wIiwgZGlyZWN0aW9uPSJuZWFyZXN0IiwgdG9sZXJhbmNlPTEwMCkKICAgIG1lcmdlZCA9IG1lcmdlZC5kcm9wbmEoKS5yZXNldF9pbmRleChkcm9wPVRydWUpCgogICAgcmV0dXJuIG1lcmdlZCwgc2NyZWVuX2RmCg==", "preprocessing/__init__.py": "IyBwcmVwcm9jZXNzaW5nIHBhY2thZ2UK", "preprocessing/feature_inventory.py": "IiIiCjIyIFVWIGZlYXR1cmUgdmVjdG9ycy4gU2VjdGlvbiA2LjEgb2YgYXJYaXY6MjMwMi4wMTc1MS4KCklNUE9SVEFOVCBDT1JSRUNUSU9OUyB2cyBuYWl2ZSBpbXBsZW1lbnRhdGlvbjoKLSBSb3RhdGlvbiBzZW5zb3Igc3RvcmVzIDMtYXhpcyByb3RhdGlvbiBWRUNUT1IgKGF6aW11dGgsIHBpdGNoLCByb2xsKQogIE5PVCBhIHF1YXRlcm5pb24uIEZpbGUgZm9ybWF0IGlzIDMgZmxvYXRzID0gc2FtZSAyMC1ieXRlIHJlY29yZCBhcyBhbGwgb3RoZXIgc2Vuc29ycy4KLSBFYXJ0aC1maXhlZCBmcmFtZSBjb252ZXJzaW9uIHVzZXMgc2NpcHkgUi5mcm9tX3JvdHZlYygpLCBub3QgUi5mcm9tX3F1YXQoKS4KLSBBTEwgMjIgZmVhdHVyZXMgYXJlIDMtY2hhbm5lbC4gQWxsIHBhZGRlZCB0byAoNCwgVCkgZm9yIHVuaWZvcm0gYnJhbmNoIGlucHV0LgotIFRvdGFsIGNoYW5uZWxzOiAyMiAqIDMgPSA2NiAobm90IDY3KS4KClJ1bjogcHl0aG9uIHByZXByb2Nlc3NpbmcvZmVhdHVyZV9pbnZlbnRvcnkucHkKRXhwZWN0ZWQ6CiAgRmVhdHVyZSBpbnZlbnRvcnk6IDIyIGZlYXR1cmVzCiAgQWxsIDIyIGFyZSAzLWNoYW5uZWwKICBUb3RhbCBjaGFubmVscyAocHJlLXBhZCk6IDY2CiAgTW9kZWwgaW5wdXQgc2hhcGUgcGVyIHNhbXBsZTogKDIyLCA0LCBUKQoiIiIKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IG5hbWVkdHVwbGUKCkZlYXR1cmVEZWYgPSBuYW1lZHR1cGxlKAogICAgIkZlYXR1cmVEZWYiLAogICAgWyJuYW1lIiwgInNvdXJjZV9zZW5zb3IiLCAidHJhbnNmb3JtIiwgImRlc2NyaXB0aW9uIl0pCgojIEFMTCBmZWF0dXJlcyBhcmUgMy1jaGFubmVsIChyb3RhdGlvbiBpcyBhemltdXRoL3BpdGNoL3JvbGwsIG5vdCBxdWF0ZXJuaW9uKQpGRUFUVVJFX0xJU1QgPSBbCiAgICAjIOKUgOKUgCBSYXcgcmVhZGluZ3MsIGRldmljZSBmcmFtZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIEZlYXR1cmVEZWYoImFjY19yYXciLCAgICAgICAgICAgImFjYyIsICAgIm5vbmUiLAogICAgICAgICAgICAgICAiUmF3IGFjY2VsZXJvbWV0ZXIsIGRldmljZSBmcmFtZSIpLAogICAgRmVhdHVyZURlZigiZ3lyb19yYXciLCAgICAgICAgICAiZ3lybyIsICAibm9uZSIsCiAgICAgICAgICAgICAgICJSYXcgZ3lyb3Njb3BlLCBkZXZpY2UgZnJhbWUiKSwKICAgIEZlYXR1cmVEZWYoIm1hZ19yYXciLCAgICAgICAgICAgIm1hZyIsICAgIm5vbmUiLAogICAgICAgICAgICAgICAiUmF3IG1hZ25ldG9tZXRlciwgZGV2aWNlIGZyYW1lIiksCiAgICAjIOKUgOKUgCBNYW51YWxseSBjb21wdXRlZCBsaW5fYWNjIChwYXBlciBTZWN0aW9uIDYuMSBmb3JtdWxhKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIEZlYXR1cmVEZWYoImxpbl9hY2NfbWFudWFsIiwgICAgImFjYyIsICAgIm1pbnVzX2dyYXYiLAogICAgICAgICAgICAgICAibGluX2FjYyA9IGFjYyAtIGdyYXYgIChkZXZpY2UgZnJhbWUpIiksCiAgICAjIOKUgOKUgCBFYXJ0aC1maXhlZCBmcmFtZSB2aWEgUi5mcm9tX3JvdHZlYyhyb3QpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgRmVhdHVyZURlZigiYWNjX3JvdCIsICAgICAgICAgICAiYWNjIiwgICAiZWFydGgiLAogICAgICAgICAgICAgICAiQWNjZWxlcm9tZXRlciBpbiBFYXJ0aC1maXhlZCBmcmFtZSIpLAogICAgRmVhdHVyZURlZigiZ3lyb19yb3QiLCAgICAgICAgICAiZ3lybyIsICAiZWFydGgiLAogICAgICAgICAgICAgICAiR3lyb3Njb3BlIGluIEVhcnRoLWZpeGVkIGZyYW1lIiksCiAgICBGZWF0dXJlRGVmKCJtYWdfcm90IiwgICAgICAgICAgICJtYWciLCAgICJlYXJ0aCIsCiAgICAgICAgICAgICAgICJNYWduZXRvbWV0ZXIgaW4gRWFydGgtZml4ZWQgZnJhbWUiKSwKICAgICMg4pSA4pSAIERpZmZlcmVuY2VzLCB1bnJvdGF0ZWQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBGZWF0dXJlRGVmKCJhY2NfZGVsdGEiLCAgICAgICAgICJhY2MiLCAgICJkaWZmIiwKICAgICAgICAgICAgICAgIkNvbnNlY3V0aXZlIGRpZmZlcmVuY2VzIG9mIHJhdyBhY2MiKSwKICAgIEZlYXR1cmVEZWYoImd5cm9fZGVsdGEiLCAgICAgICAgImd5cm8iLCAgImRpZmYiLAogICAgICAgICAgICAgICAiQ29uc2VjdXRpdmUgZGlmZmVyZW5jZXMgb2YgcmF3IGd5cm8iKSwKICAgIEZlYXR1cmVEZWYoIm1hZ19kZWx0YSIsICAgICAgICAgIm1hZyIsICAgImRpZmYiLAogICAgICAgICAgICAgICAiQ29uc2VjdXRpdmUgZGlmZmVyZW5jZXMgb2YgcmF3IG1hZyIpLAogICAgIyDilIDilIAgRGlmZmVyZW5jZXMsIHJvdGF0ZWQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBGZWF0dXJlRGVmKCJhY2Nfcm90X2RlbHRhIiwgICAgICJhY2MiLCAgICJlYXJ0aCtkaWZmIiwKICAgICAgICAgICAgICAgIkNvbnNlY3V0aXZlIGRpZmZlcmVuY2VzIG9mIHJvdGF0ZWQgYWNjIiksCiAgICBGZWF0dXJlRGVmKCJneXJvX3JvdF9kZWx0YSIsICAgICJneXJvIiwgICJlYXJ0aCtkaWZmIiwKICAgICAgICAgICAgICAgIkNvbnNlY3V0aXZlIGRpZmZlcmVuY2VzIG9mIHJvdGF0ZWQgZ3lybyIpLAogICAgRmVhdHVyZURlZigibWFnX3JvdF9kZWx0YSIsICAgICAibWFnIiwgICAiZWFydGgrZGlmZiIsCiAgICAgICAgICAgICAgICJDb25zZWN1dGl2ZSBkaWZmZXJlbmNlcyBvZiByb3RhdGVkIG1hZyIpLAogICAgIyDilIDilIAgSW50ZWdyYWxzLCB1bnJvdGF0ZWQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBGZWF0dXJlRGVmKCJhY2NfaW50ZWdyYWwiLCAgICAgICJhY2MiLCAgICJjdW1zdW0iLAogICAgICAgICAgICAgICAiQ3VtdWxhdGl2ZSBpbnRlZ3JhbCBvZiByYXcgYWNjIiksCiAgICBGZWF0dXJlRGVmKCJneXJvX2ludGVncmFsIiwgICAgICJneXJvIiwgICJjdW1zdW0iLAogICAgICAgICAgICAgICAiQ3VtdWxhdGl2ZSBpbnRlZ3JhbCBvZiByYXcgZ3lybyIpLAogICAgRmVhdHVyZURlZigibWFnX2ludGVncmFsIiwgICAgICAibWFnIiwgICAiY3Vtc3VtIiwKICAgICAgICAgICAgICAgIkN1bXVsYXRpdmUgaW50ZWdyYWwgb2YgcmF3IG1hZyIpLAogICAgIyDilIDilIAgSW50ZWdyYWxzLCByb3RhdGVkIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgRmVhdHVyZURlZigiYWNjX3JvdF9pbnRlZ3JhbCIsICAiYWNjIiwgICAiZWFydGgrY3Vtc3VtIiwKICAgICAgICAgICAgICAgIkN1bXVsYXRpdmUgaW50ZWdyYWwgb2Ygcm90YXRlZCBhY2MiKSwKICAgIEZlYXR1cmVEZWYoImd5cm9fcm90X2ludGVncmFsIiwgImd5cm8iLCAgImVhcnRoK2N1bXN1bSIsCiAgICAgICAgICAgICAgICJDdW11bGF0aXZlIGludGVncmFsIG9mIHJvdGF0ZWQgZ3lybyIpLAogICAgRmVhdHVyZURlZigibWFnX3JvdF9pbnRlZ3JhbCIsICAibWFnIiwgICAiZWFydGgrY3Vtc3VtIiwKICAgICAgICAgICAgICAgIkN1bXVsYXRpdmUgaW50ZWdyYWwgb2Ygcm90YXRlZCBtYWciKSwKICAgICMg4pSA4pSAIGxpbl9hY2MgZGVsdGEgKyBpbnRlZ3JhbCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIEZlYXR1cmVEZWYoImxpbl9hY2NfZGVsdGEiLCAgICAgImFjYyIsICAgIm1pbnVzX2dyYXYrZGlmZiIsCiAgICAgICAgICAgICAgICJDb25zZWN1dGl2ZSBkaWZmZXJlbmNlcyBvZiBsaW5fYWNjIiksCiAgICBGZWF0dXJlRGVmKCJsaW5fYWNjX2ludGVncmFsIiwgICJhY2MiLCAgICJtaW51c19ncmF2K2N1bXN1bSIsCiAgICAgICAgICAgICAgICJDdW11bGF0aXZlIGludGVncmFsIG9mIGxpbl9hY2MiKSwKICAgICMg4pSA4pSAIFJhdyByb3RhdGlvbiB2ZWN0b3IgKGF6aW11dGgsIHBpdGNoLCByb2xsKSDigJQgMy1heGlzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgRmVhdHVyZURlZigicm90X3JhdyIsICAgICAgICAgICAicm90IiwgICAibm9uZSIsCiAgICAgICAgICAgICAgICJSYXcgcm90YXRpb24gdmVjdG9yIChhemltdXRoLCBwaXRjaCwgcm9sbCkiKSwKXQoKTl9GRUFUVVJFUyAgICA9IDIyCk5fQ0hBTk5FTFMgICAgPSAzICAgIyBhbGwgZmVhdHVyZXMgYXJlIDMtY2hhbm5lbApNT0RFTF9JTlBVVF9DID0gNCAgICMgcGFkZGVkIHRvIDQgZm9yIHVuaWZvcm0gYnJhbmNoIGlucHV0Cgphc3NlcnQgbGVuKEZFQVRVUkVfTElTVCkgPT0gTl9GRUFUVVJFUwoKCmRlZiBnZXRfZmVhdHVyZV9zaGFwZXMoVDogaW50ID0gNTApOgogICAgIiIiUmV0dXJucyBsaXN0IG9mIChNT0RFTF9JTlBVVF9DLCBUKSBmb3IgZWFjaCBvZiB0aGUgMjIgZmVhdHVyZXMuIiIiCiAgICByZXR1cm4gWyhNT0RFTF9JTlBVVF9DLCBUKV0gKiBOX0ZFQVRVUkVTCgoKZGVmIHZlcmlmeSgpOgogICAgYXNzZXJ0IGxlbihGRUFUVVJFX0xJU1QpID09IDIyCiAgICBwcmludChmIkZlYXR1cmUgaW52ZW50b3J5OiB7bGVuKEZFQVRVUkVfTElTVCl9IGZlYXR1cmVzIikKICAgIHByaW50KGYiQWxsIDIyIGFyZSB7Tl9DSEFOTkVMU30tY2hhbm5lbCIpCiAgICBwcmludChmIlRvdGFsIGNoYW5uZWxzIChwcmUtcGFkKToge05fRkVBVFVSRVMgKiBOX0NIQU5ORUxTfSIpCiAgICBwcmludChmIk1vZGVsIGlucHV0IHNoYXBlIHBlciBzYW1wbGU6ICgyMiwge01PREVMX0lOUFVUX0N9LCBUKSIpCiAgICBwcmludCgpCiAgICBwcmludChmInsnSWR4Jzo+M30gIHsnTmFtZSc6PDIyfSB7J1NvdXJjZSc6PDZ9IHsnVHJhbnNmb3JtJzo8MTh9IikKICAgIHByaW50KCItIiAqIDU2KQogICAgZm9yIGksIGYgaW4gZW51bWVyYXRlKEZFQVRVUkVfTElTVCk6CiAgICAgICAgcHJpbnQoZiIgIHtpKzE6MmR9LiB7Zi5uYW1lOjwyMn0ge2Yuc291cmNlX3NlbnNvcjo8Nn0ge2YudHJhbnNmb3JtfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHZlcmlmeSgpCg==", "preprocessing/mpi_preprocess.py": "IiIiCk1QSSBkYXRhc2V0IHByZXByb2Nlc3NpbmcuClBhcGVyOiBTZWN0aW9ucyA0LjEuMSBhbmQgNC4zLjEgb2YgYXJYaXY6MjMwMi4wMTc1MQoKRUZGSUNJRU5UIEFQUFJPQUNIOiByZWFkcyBvbmx5IHNtYWxsIHRpbWUgd2luZG93cyBhcm91bmQgc2NyZWVuIGV2ZW50cywKTk9UIHRoZSBlbnRpcmUgMTItd2VlayBzZXNzaW9uICh3aGljaCB3b3VsZCBiZSB+MjVNIHJvd3MgcGVyIHNlbnNvcikuCgpSdW46IHB5dGhvbiBwcmVwcm9jZXNzaW5nL21waV9wcmVwcm9jZXNzLnB5IC0tdXNlLWR1bW15CiIiIgoKaW1wb3J0IG9zLCBzeXMsIGFyZ3BhcnNlCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkgaW1wb3J0IGludGVycG9sYXRlCgpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpCmZyb20gY29uZmlncy5jb25maWcgaW1wb3J0IGNmZwpmcm9tIHV0aWxzLmJpbl9yZWFkZXIgaW1wb3J0IChyZWFkX3NlbnNvcl9iaW4sIHJlYWRfc2NyZWVuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgU0VOU09SX0ZJTEVTLCBTRU5TT1JfQ09MUywgQUxMX1NFTlNPUl9DT0xTLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgRkxBR19VU0VSX1BSRVNFTlQsIEZMQUdfU0NSRUVOX09GRiwgRkxBR19TQ1JFRU5fT04pCgpNUElfSU5QVVRfRElSUyA9IFsKICAgICIva2FnZ2xlL2lucHV0L2RhdGFzZXRzL2RqYWFyZi9tb3Rpb25pZC1pbXUtYWxsLW1vdGlvbnMtcGFydDEvSU1VX2FsbF9tb3Rpb25zX3BhcnQxIiwKICAgICIva2FnZ2xlL2lucHV0L2RhdGFzZXRzL2RqYWFyZi9tb3Rpb25pZC1pbXUtYWxsLW1vdGlvbnMtcGFydDIvSU1VX2FsbF9tb3Rpb25zX3BhcnQyIiwKICAgICIva2FnZ2xlL2lucHV0L2RhdGFzZXRzL2RqYWFyZi9tb3Rpb25pZC1pbXUtYWxsLW1vdGlvbnMtcGFydDMvSU1VX2FsbF9tb3Rpb25zX3BhcnQzIiwKXQpQUk9DRVNTRURfRElSID0gImRhdGEvbXBpL3Byb2Nlc3NlZCIKTElOX0NPTFMgICAgICA9IFsibGluX1giLCAibGluX1kiLCAibGluX1oiXQoKCmRlZiByZWFkX3NlbnNvcnNfd2luZG93KHNlc3Npb25fZGlyOiBzdHIsIHRfc3RhcnQ6IGludCwgdF9lbmQ6IGludCkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUmVhZCBhbGwgNiBzZW5zb3JzIGluIGEgdGlnaHQgdGltZSB3aW5kb3cgYW5kIG1lcmdlLiIiIgogICAgc2Vuc29yX2RmcyA9IHt9CiAgICBmb3IgbmFtZSwgZm5hbWUgaW4gU0VOU09SX0ZJTEVTLml0ZW1zKCk6CiAgICAgICAgZnBhdGggPSBvcy5wYXRoLmpvaW4oc2Vzc2lvbl9kaXIsIGZuYW1lKQogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhmcGF0aCk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZGYgPSByZWFkX3NlbnNvcl9iaW4oZnBhdGgsIHRfc3RhcnQ9dF9zdGFydCwgdF9lbmQ9dF9lbmQpCiAgICAgICAgaWYgbGVuKGRmKSA9PSAwOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGNvbHMgPSBTRU5TT1JfQ09MU1tuYW1lXQogICAgICAgIGRmID0gZGYucmVuYW1lKGNvbHVtbnM9eyJYIjogY29sc1swXSwgIlkiOiBjb2xzWzFdLCAiWiI6IGNvbHNbMl19KQogICAgICAgIHNlbnNvcl9kZnNbbmFtZV0gPSBkZi5zb3J0X3ZhbHVlcygidGltZXN0YW1wIikucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKICAgIG1lcmdlZCA9IHNlbnNvcl9kZnNbImFjYyJdCiAgICBmb3IgbmFtZSBpbiBbImdyYXYiLCAiZ3lybyIsICJsaW4iLCAibWFnIiwgInJvdCJdOgogICAgICAgIG1lcmdlZCA9IHBkLm1lcmdlX2Fzb2YoCiAgICAgICAgICAgIG1lcmdlZCwgc2Vuc29yX2Rmc1tuYW1lXSwKICAgICAgICAgICAgb249InRpbWVzdGFtcCIsIGRpcmVjdGlvbj0ibmVhcmVzdCIsIHRvbGVyYW5jZT0xMDApCiAgICByZXR1cm4gbWVyZ2VkLmRyb3BuYSgpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgZXh0cmFjdF9wb3NpdGl2ZXMoc2Vzc2lvbl9kaXI6IHN0ciwgc2NyZWVuX2RmOiBwZC5EYXRhRnJhbWUpIC0+IGxpc3Q6CiAgICAiIiJQYXBlciA0LjMuMTogMyBzZWMgYmVmb3JlIGVhY2ggVVNFUl9QUkVTRU5ULCA+PTEwMCByZWFkaW5ncy4iIiIKICAgIHNhbXBsZXMgPSBbXQogICAgd2luZG93X21zID0gaW50KGNmZy5tcGlfd2luZG93X3NlYyAqIDEwMDApCiAgICBmb3IgXywgcm93IGluIHNjcmVlbl9kZltzY3JlZW5fZGZbImV2ZW50Il0gPT0gRkxBR19VU0VSX1BSRVNFTlRdLml0ZXJyb3dzKCk6CiAgICAgICAgdHMgPSBpbnQocm93WyJ0aW1lc3RhbXAiXSkKICAgICAgICBtZXJnZWQgPSByZWFkX3NlbnNvcnNfd2luZG93KHNlc3Npb25fZGlyLCB0cyAtIHdpbmRvd19tcywgdHMpCiAgICAgICAgaWYgbWVyZ2VkIGlzIG5vdCBOb25lIGFuZCBsZW4obWVyZ2VkKSA+PSBjZmcubXBpX21pbl9yZWFkaW5nczoKICAgICAgICAgICAgc2FtcGxlcy5hcHBlbmQobWVyZ2VkW0FMTF9TRU5TT1JfQ09MU10udmFsdWVzLmFzdHlwZShucC5mbG9hdDMyKSkKICAgIHJldHVybiBzYW1wbGVzCgoKZGVmIGV4dHJhY3RfbmVnYXRpdmVzKHNlc3Npb25fZGlyOiBzdHIsIHNjcmVlbl9kZjogcGQuRGF0YUZyYW1lKSAtPiBsaXN0OgogICAgIiIiUGFwZXIgNC4zLjE6IFNDUkVFTl9PRkYgdG8gbmV4dCBTQ1JFRU5fT04vVVNFUl9QUkVTRU5ULCBleGNsdWRlIGxhc3QgM3MuCiAgICBNRU1PUlkgRklYOiBvbmx5IHJlYWQgbGFzdCA2MHMgb2YgZWFjaCBpbnRlcnZhbCAobm90IGhvdXJzIG9mIGlkbGUgZGF0YSkuIiIiCiAgICBzYW1wbGVzID0gW10KICAgIHdpbmRvd19tcyA9IGludChjZmcubXBpX3dpbmRvd19zZWMgKiAxMDAwKQogICAgbWF4X3JlYWRfc2VjID0gNjAgICMgY2FwOiBkb24ndCByZWFkIGhvdXJzIG9mIGlkbGUgcGhvbmUgZGF0YQogICAgbWF4X3JlYWRfbXMgPSBtYXhfcmVhZF9zZWMgKiAxMDAwCiAgICBvZmZfZXZlbnRzID0gc2NyZWVuX2RmW3NjcmVlbl9kZlsiZXZlbnQiXSA9PSBGTEFHX1NDUkVFTl9PRkZdCgogICAgZm9yIF8sIHJvdyBpbiBvZmZfZXZlbnRzLml0ZXJyb3dzKCk6CiAgICAgICAgb2ZmX3RzID0gaW50KHJvd1sidGltZXN0YW1wIl0pCiAgICAgICAgbGF0ZXIgPSBzY3JlZW5fZGZbCiAgICAgICAgICAgIChzY3JlZW5fZGZbInRpbWVzdGFtcCJdID4gb2ZmX3RzKSAmCiAgICAgICAgICAgIChzY3JlZW5fZGZbImV2ZW50Il0uaXNpbihbRkxBR19TQ1JFRU5fT04sIEZMQUdfVVNFUl9QUkVTRU5UXSkpXQogICAgICAgIGlmIGxhdGVyLmVtcHR5OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGVuZF90cyA9IGludChsYXRlci5pbG9jWzBdWyJ0aW1lc3RhbXAiXSkKICAgICAgICBlZmZlY3RpdmVfZW5kID0gZW5kX3RzIC0gd2luZG93X21zCiAgICAgICAgaWYgZWZmZWN0aXZlX2VuZCA8PSBvZmZfdHM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBNRU1PUlkgRklYOiBvbmx5IHJlYWQgbGFzdCA2MHMsIG5vdCBlbnRpcmUgZ2FwCiAgICAgICAgcmVhZF9zdGFydCA9IG1heChvZmZfdHMsIGVmZmVjdGl2ZV9lbmQgLSBtYXhfcmVhZF9tcykKICAgICAgICBpZiBlZmZlY3RpdmVfZW5kIC0gcmVhZF9zdGFydCA8IHdpbmRvd19tczoKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgbWVyZ2VkID0gcmVhZF9zZW5zb3JzX3dpbmRvdyhzZXNzaW9uX2RpciwgcmVhZF9zdGFydCwgZWZmZWN0aXZlX2VuZCkKICAgICAgICBpZiBtZXJnZWQgaXMgTm9uZSBvciBsZW4obWVyZ2VkKSA8IGNmZy5tcGlfbWluX3JlYWRpbmdzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgICMgQ2hlY2sgbm90IHN0YXRpb25hcnkKICAgICAgICBsaW4gPSBtZXJnZWRbTElOX0NPTFNdLnZhbHVlcwogICAgICAgIGlmIG5wLmFsbChucC5hYnMobGluKSA8IGNmZy5zdGF0aW9uYXJ5X3RocmVzaG9sZCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgc2FtcGxlcy5hcHBlbmQobWVyZ2VkW0FMTF9TRU5TT1JfQ09MU10udmFsdWVzLmFzdHlwZShucC5mbG9hdDMyKSkKICAgIHJldHVybiBzYW1wbGVzCgoKZGVmIG5vcm1hbGl6ZV9sZW5ndGgoc2FtcGxlOiBucC5uZGFycmF5LCB0YXJnZXRfbGVuOiBpbnQgPSAxNTApIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIobiwgMTgpIC0+ICgxOCwgdGFyZ2V0X2xlbikgY2hhbm5lbC1maXJzdC4iIiIKICAgIG4sIGMgPSBzYW1wbGUuc2hhcGUKICAgIGlmIG4gPT0gdGFyZ2V0X2xlbjoKICAgICAgICByZXR1cm4gc2FtcGxlLlQuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBpZiBuIDwgdGFyZ2V0X2xlbjoKICAgICAgICB4X29sZCwgeF9uZXcgPSBucC5saW5zcGFjZSgwLCAxLCBuKSwgbnAubGluc3BhY2UoMCwgMSwgdGFyZ2V0X2xlbikKICAgICAgICBvdXQgPSBucC56ZXJvcygodGFyZ2V0X2xlbiwgYyksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoYyk6CiAgICAgICAgICAgIG91dFs6LCBpXSA9IGludGVycG9sYXRlLmludGVycDFkKAogICAgICAgICAgICAgICAgeF9vbGQsIHNhbXBsZVs6LCBpXSwga2luZD0ibGluZWFyIikoeF9uZXcpCiAgICAgICAgcmV0dXJuIG91dC5UCiAgICByZXR1cm4gc2FtcGxlWy10YXJnZXRfbGVuOl0uVC5hc3R5cGUobnAuZmxvYXQzMikKCgpkZWYgZGlzY292ZXJfc2Vzc2lvbnMoaW5wdXRfZGlyczogbGlzdCkgLT4gbGlzdDoKICAgICIiIlJldHVybnMgbGlzdCBvZiAodXNlcl9pZCwgZGV2aWNlX2lkLCBzZXNzaW9uX2RpcikuIiIiCiAgICBzZXNzaW9ucyA9IFtdCiAgICBmb3Igcm9vdCBpbiBpbnB1dF9kaXJzOgogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhyb290KToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgdXNlciBpbiBzb3J0ZWQob3MubGlzdGRpcihyb290KSk6CiAgICAgICAgICAgIHVzZXJfcGF0aCA9IG9zLnBhdGguam9pbihyb290LCB1c2VyKQogICAgICAgICAgICBpZiBub3Qgb3MucGF0aC5pc2Rpcih1c2VyX3BhdGgpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZm9yIHBob25lIGluIHNvcnRlZChvcy5saXN0ZGlyKHVzZXJfcGF0aCkpOgogICAgICAgICAgICAgICAgcGhvbmVfcGF0aCA9IG9zLnBhdGguam9pbih1c2VyX3BhdGgsIHBob25lKQogICAgICAgICAgICAgICAgaWYgbm90IG9zLnBhdGguaXNkaXIocGhvbmVfcGF0aCk6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGZvciBzZXNzaW9uIGluIHNvcnRlZChvcy5saXN0ZGlyKHBob25lX3BhdGgpKToKICAgICAgICAgICAgICAgICAgICBzZXNzaW9uX3BhdGggPSBvcy5wYXRoLmpvaW4ocGhvbmVfcGF0aCwgc2Vzc2lvbikKICAgICAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKHNlc3Npb25fcGF0aCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlc3Npb25zLmFwcGVuZCgodXNlciwgcGhvbmUsIHNlc3Npb25fcGF0aCkpCiAgICByZXR1cm4gc2Vzc2lvbnMKCgpkZWYgcHJvY2Vzc19zZXNzaW9uKHVpZDogc3RyLCBkaWQ6IHN0ciwgc2Vzc2lvbl9kaXI6IHN0cik6CiAgICB0YXJnZXRfbGVuID0gaW50KGNmZy5tcGlfc2FtcGxpbmdfcmF0ZSAqIGNmZy5tcGlfd2luZG93X3NlYykKCiAgICAjIFJlYWQgc2NyZWVuIGV2ZW50cyAoc21hbGwgdGV4dCBmaWxlKQogICAgc2NyZWVuX3BhdGggPSBvcy5wYXRoLmpvaW4oc2Vzc2lvbl9kaXIsICJzY3JlZW4udHh0IikKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhzY3JlZW5fcGF0aCk6CiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUKICAgIHNjcmVlbl9kZiA9IHJlYWRfc2NyZWVuKHNjcmVlbl9wYXRoKQogICAgaWYgbGVuKHNjcmVlbl9kZikgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZSwgTm9uZQoKICAgICMgRXh0cmFjdCB3aW5kb3dzIGFyb3VuZCBldmVudHMgKHJlYWRzIG9ubHkgc21hbGwgY2h1bmtzKQogICAgcG9zID0gZXh0cmFjdF9wb3NpdGl2ZXMoc2Vzc2lvbl9kaXIsIHNjcmVlbl9kZikKICAgIG5lZyA9IGV4dHJhY3RfbmVnYXRpdmVzKHNlc3Npb25fZGlyLCBzY3JlZW5fZGYpCgogICAgaWYgbGVuKHBvcykgPCAxMCBvciBsZW4obmVnKSA8IDEwOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCgogICAgWCA9IG5wLnN0YWNrKFtub3JtYWxpemVfbGVuZ3RoKHMsIHRhcmdldF9sZW4pIGZvciBzIGluIHBvcyArIG5lZ10pCiAgICB5ID0gbnAuYXJyYXkoWzFdKmxlbihwb3MpICsgWzBdKmxlbihuZWcpLCBkdHlwZT1ucC5pbnQ2NCkKICAgIHJldHVybiBYLCB5CgoKZGVmIHJ1bl9vbl9kdW1teSgpOgogICAgZnJvbSB1dGlscy5kdW1teV9kYXRhIGltcG9ydCBsb2FkX21waV9kdW1teQogICAgWCwgeSwgXyA9IGxvYWRfbXBpX2R1bW15KCkKICAgIHByaW50KGYiRHVtbXkgTVBJOiBYPXtYLnNoYXBlfSwgeT17eS5zaGFwZX0iKQogICAgZm9yIG5faW4gaW4gWzEyMCwgMjAwLCAxNTBdOgogICAgICAgIG91dCA9IG5vcm1hbGl6ZV9sZW5ndGgobnAucmFuZG9tLnJhbmRuKG5faW4sIDE4KS5hc3R5cGUobnAuZmxvYXQzMiksIDE1MCkKICAgICAgICBhc3NlcnQgb3V0LnNoYXBlID09ICgxOCwgMTUwKSwgZiJHb3Qge291dC5zaGFwZX0iCiAgICAgICAgcHJpbnQoZiIgIG5vcm1hbGl6ZV9sZW5ndGgoe25faW59LCAxOCkgLT4ge291dC5zaGFwZX0gIE9LIikKICAgIHByaW50KCJEdW1teSBNUEkgdGVzdCBwYXNzZWQuIikKCgpkZWYgbWFpbih1c2VfZHVtbXk9RmFsc2UpOgogICAgaWYgdXNlX2R1bW15OgogICAgICAgIHJ1bl9vbl9kdW1teSgpOyByZXR1cm4KICAgIG9zLm1ha2VkaXJzKFBST0NFU1NFRF9ESVIsIGV4aXN0X29rPVRydWUpCiAgICBzZXNzaW9ucyA9IGRpc2NvdmVyX3Nlc3Npb25zKE1QSV9JTlBVVF9ESVJTKQogICAgcHJpbnQoZiJGb3VuZCB7bGVuKHNlc3Npb25zKX0gc2Vzc2lvbnMgYWNyb3NzIDMgTVBJIGRhdGFzZXRzLiIpCiAgICByb3dzID0gW10KICAgIGZvciB1aWQsIGRpZCwgc2RpciBpbiBzZXNzaW9uczoKICAgICAgICBwcmludChmIiAgdXNlcj17dWlkfSBkZXZpY2U9e2RpZH0uLi4iLCBlbmQ9IiAiLCBmbHVzaD1UcnVlKQogICAgICAgIFgsIHkgPSBwcm9jZXNzX3Nlc3Npb24odWlkLCBkaWQsIHNkaXIpCiAgICAgICAga2V5ICA9IGYie3VpZH1fe2RpZH0iCiAgICAgICAgaWYgWCBpcyBOb25lOgogICAgICAgICAgICBwcmludCgiTi9BIikKICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJ1c2VyX2lkIjogdWlkLCAiZGV2aWNlX2lkIjogZGlkLAogICAgICAgICAgICAgICAgICAgICAgICAgIm5fcG9zIjogMCwgIm5fbmVnIjogMCwgInN0YXR1cyI6ICJOL0EifSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBucC5zYXZleihvcy5wYXRoLmpvaW4oUFJPQ0VTU0VEX0RJUiwgZiJ7a2V5fS5ucHoiKSwgWD1YLCB5PXkpCiAgICAgICAgICAgIG5fcG9zLCBuX25lZyA9IGludCgoeT09MSkuc3VtKCkpLCBpbnQoKHk9PTApLnN1bSgpKQogICAgICAgICAgICBwcmludChmIlg9e1guc2hhcGV9LCBwb3M9e25fcG9zfSwgbmVnPXtuX25lZ30iKQogICAgICAgICAgICByb3dzLmFwcGVuZCh7InVzZXJfaWQiOiB1aWQsICJkZXZpY2VfaWQiOiBkaWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAibl9wb3MiOiBuX3BvcywgIm5fbmVnIjogbl9uZWcsICJzdGF0dXMiOiAiT0sifSkKICAgIG1mID0gcGQuRGF0YUZyYW1lKHJvd3MpCiAgICBtZi50b19jc3Yob3MucGF0aC5qb2luKFBST0NFU1NFRF9ESVIsICJtYW5pZmVzdC5jc3YiKSwgaW5kZXg9RmFsc2UpCiAgICBwcmludChmIkRvbmUuIHsobWYuc3RhdHVzPT0nT0snKS5zdW0oKX0gdmFsaWQgc2Vzc2lvbnMuIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS11c2UtZHVtbXkiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIG1haW4odXNlX2R1bW15PWFyZ3MudXNlX2R1bW15KQo=", "preprocessing/uv_preprocess.py": "IiIiClVWIGRhdGFzZXQgcHJlcHJvY2Vzc2luZy4KUGFwZXI6IFNlY3Rpb25zIDQuMS4yIGFuZCA0LjMuMiBvZiBhclhpdjoyMzAyLjAxNzUxCgpJbnB1dDogS2FnZ2xlIGRhdGFzZXQgbW91bnRlZCBhdDoKICAva2FnZ2xlL2lucHV0L21vdGlvbmlkLWltdS1zcGVjaWZpYy1tb3Rpb24vSU1VX3NwZWNpZmljX21vdGlvbi90cmFpbl92YWxfdGVzdC8KCkRpcmVjdG9yeSBzdHJ1Y3R1cmU6CiAge3VzZXJfaWR9L3MyMC97dXNlcl9pZH1fMjAwMDAvCiAgICBhY2NlbC5iaW4sIGdyYXZpdHkuYmluLCBneXJvLmJpbiwgbGluQWNjZWwuYmluLCBNYWduZXRpY0ZpZWxkLmJpbiwKICAgIFJvdGF0aW9uLmJpbiwgc2NyZWVuLnR4dAoKUnVuOiBweXRob24gcHJlcHJvY2Vzc2luZy91dl9wcmVwcm9jZXNzLnB5IC0tdXNlLWR1bW15CiIiIgoKaW1wb3J0IG9zLCBzeXMsIGFyZ3BhcnNlCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkuc3BhdGlhbC50cmFuc2Zvcm0gaW1wb3J0IFJvdGF0aW9uIGFzIFIKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkKZnJvbSBjb25maWdzLmNvbmZpZyBpbXBvcnQgY2ZnCmZyb20gdXRpbHMuYmluX3JlYWRlciBpbXBvcnQgbG9hZF9zZXNzaW9uLCBGTEFHX1VTRVJfUFJFU0VOVApmcm9tIHByZXByb2Nlc3NpbmcuZmVhdHVyZV9pbnZlbnRvcnkgaW1wb3J0IEZFQVRVUkVfTElTVAoKVVZfSU5QVVRfRElSICA9ICgiL2thZ2dsZS9pbnB1dC9kYXRhc2V0cy9kamFhcmYvbW90aW9uaWQtaW11LXNwZWNpZmljLW1vdGlvbiIKICAgICAgICAgICAgICAgICAiL0lNVV9zcGVjaWZpY19tb3Rpb24vdHJhaW5fdmFsX3Rlc3QiKQpQUk9DRVNTRURfRElSID0gImRhdGEvdXYvcHJvY2Vzc2VkIgoKCmRlZiByb3RhdGVfdG9fZWFydGgodjogbnAubmRhcnJheSwgcm90X3ZlYzogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICIiIgogICAgUGFwZXIgU2VjdGlvbiA2LjE6IGNvbnZlcnQgZGV2aWNlLWZyYW1lIHJlYWRpbmdzIHRvIEVhcnRoLWZpeGVkIGZyYW1lLgogICAgdjogICAgICAgKFQsIDMpIOKAlCBzZW5zb3IgcmVhZGluZ3MgaW4gZGV2aWNlIGZyYW1lCiAgICByb3RfdmVjOiAoVCwgMykg4oCUIHJvdGF0aW9uIHZlY3RvciAoYXppbXV0aCwgcGl0Y2gsIHJvbGwpIGZyb20gUm90YXRpb24gc2Vuc29yCiAgICBSZXR1cm5zOiAoVCwgMykgaW4gRWFydGgtZml4ZWQgZnJhbWUKICAgIFVzZXMgc2NpcHkgUi5mcm9tX3JvdHZlYygpIOKAlCBoYW5kbGVzIHNtYWxsL3plcm8gbm9ybXMgYXV0b21hdGljYWxseS4KICAgICIiIgogICAgcmV0dXJuIFIuZnJvbV9yb3R2ZWMocm90X3ZlYykuYXBwbHkodikuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGV4dHJhY3Rfd2luZG93KG1lcmdlZF9kZjogcGQuRGF0YUZyYW1lLCB1bmxvY2tfdHM6IGludCk6CiAgICAiIiIKICAgIFBhcGVyIDQuMy4yOiBsYXN0IDEgc2Vjb25kIG9mIGRhdGEgYmVmb3JlIHVubG9jay4KICAgIFJldHVybnMgRGF0YUZyYW1lIG9mIGV4YWN0bHkgdXZfc2FtcGxpbmdfcmF0ZSByb3dzLCBvciBOb25lLgogICAgIiIiCiAgICBuICAgICAgICAgPSBpbnQoY2ZnLnV2X3dpbmRvd19zZWMgKiBjZmcudXZfc2FtcGxpbmdfcmF0ZSkKICAgIHdpbmRvd19tcyA9IGludChjZmcudXZfd2luZG93X3NlYyAqIDEwMDApCiAgICBzdWIgICAgICAgPSAobWVyZ2VkX2RmWwogICAgICAgICAgICAgICAgICAgIChtZXJnZWRfZGZbInRpbWVzdGFtcCJdIDw9IHVubG9ja190cykgJgogICAgICAgICAgICAgICAgICAgIChtZXJnZWRfZGZbInRpbWVzdGFtcCJdID4gIHVubG9ja190cyAtIHdpbmRvd19tcyldCiAgICAgICAgICAgICAgICAgLnNvcnRfdmFsdWVzKCJ0aW1lc3RhbXAiKQogICAgICAgICAgICAgICAgIC50YWlsKG4pKQogICAgcmV0dXJuIHN1Yi5yZXNldF9pbmRleChkcm9wPVRydWUpIGlmIGxlbihzdWIpID09IG4gZWxzZSBOb25lCgoKZGVmIHBhZDQoeDogbnAubmRhcnJheSwgVDogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiKFQsIDMpIOKGkiAoNCwgVCk6IHplcm8tcGFkIDR0aCBjaGFubmVsLiIiIgogICAgb3V0ICAgICAgICA9IG5wLnplcm9zKCg0LCBUKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIG91dFs6MywgOl0gPSB4LlQKICAgIHJldHVybiBvdXQKCgpkZWYgZGlmZih4OiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiQ29uc2VjdXRpdmUgZGlmZmVyZW5jZXMsIHByZXNlcnZpbmcgbGVuZ3RoIFQuIiIiCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoW3hbOjFdLCBucC5kaWZmKHgsIGF4aXM9MCldLCBheGlzPTApCgoKZGVmIGNvbXB1dGVfZmVhdHVyZXMod2luZG93OiBwZC5EYXRhRnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIKICAgIFBhcGVyIFNlY3Rpb24gNi4xOiAyMiBmZWF0dXJlIHZlY3RvcnMuCiAgICBSZXR1cm5zIHNoYXBlICgyMiwgNCwgVCkuCiAgICBBbGwgZmVhdHVyZXMgYXJlIDMtY2hhbm5lbCwgemVyby1wYWRkZWQgdG8gNCBmb3IgdW5pZm9ybSBicmFuY2ggaW5wdXQuCiAgICBGZWF0dXJlIG9yZGVyIG1hdGNoZXMgRkVBVFVSRV9MSVNUIGV4YWN0bHkuCgogICAgRWFydGgtZml4ZWQgZnJhbWUgdXNlcyBSLmZyb21fcm90dmVjKHJvdCkgd2hlcmUgcm90IGlzIHRoZQogICAgcm90YXRpb24gdmVjdG9yIChhemltdXRoLCBwaXRjaCwgcm9sbCkgZnJvbSB0aGUgQW5kcm9pZCBSb3RhdGlvbiBzZW5zb3IuCiAgICAiIiIKICAgIFQgICAgPSBsZW4od2luZG93KQogICAgYWNjICA9IHdpbmRvd1tbImFjY19YIiwgICJhY2NfWSIsICAiYWNjX1oiXV0udmFsdWVzLmFzdHlwZShucC5mbG9hdDMyKQogICAgZ3JhdiA9IHdpbmRvd1tbImdyYXZfWCIsICJncmF2X1kiLCAiZ3Jhdl9aIl1dLnZhbHVlcy5hc3R5cGUobnAuZmxvYXQzMikKICAgIGd5cm8gPSB3aW5kb3dbWyJneXJvX1giLCAiZ3lyb19ZIiwgImd5cm9fWiJdXS52YWx1ZXMuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBsaW4gID0gd2luZG93W1sibGluX1giLCAgImxpbl9ZIiwgICJsaW5fWiJdXS52YWx1ZXMuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBtYWcgID0gd2luZG93W1sibWFnX1giLCAgIm1hZ19ZIiwgICJtYWdfWiJdXS52YWx1ZXMuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICByb3QgID0gd2luZG93W1sicm90X1giLCAgInJvdF9ZIiwgICJyb3RfWiJdXS52YWx1ZXMuYXN0eXBlKG5wLmZsb2F0MzIpCgogICAgbGluX2FjYyAgPSBhY2MgLSBncmF2ICAgICAgICAgICAgICAgICAgICAgICAjIHBhcGVyIFNlY3Rpb24gNi4xIGZvcm11bGEKICAgIGFjY19yb3QgID0gcm90YXRlX3RvX2VhcnRoKGFjYywgIHJvdCkKICAgIGd5cm9fcm90ID0gcm90YXRlX3RvX2VhcnRoKGd5cm8sIHJvdCkKICAgIG1hZ19yb3QgID0gcm90YXRlX3RvX2VhcnRoKG1hZywgIHJvdCkKCiAgICByZXR1cm4gbnAuc3RhY2soWwogICAgICAgIHBhZDQoYWNjLCAgICAgICAgICAgICAgICAgICAgICAgIFQpLCAgIyAgMSAgYWNjX3JhdwogICAgICAgIHBhZDQoZ3lybywgICAgICAgICAgICAgICAgICAgICAgIFQpLCAgIyAgMiAgZ3lyb19yYXcKICAgICAgICBwYWQ0KG1hZywgICAgICAgICAgICAgICAgICAgICAgICBUKSwgICMgIDMgIG1hZ19yYXcKICAgICAgICBwYWQ0KGxpbl9hY2MsICAgICAgICAgICAgICAgICAgICBUKSwgICMgIDQgIGxpbl9hY2NfbWFudWFsCiAgICAgICAgcGFkNChhY2Nfcm90LCAgICAgICAgICAgICAgICAgICAgVCksICAjICA1ICBhY2Nfcm90CiAgICAgICAgcGFkNChneXJvX3JvdCwgICAgICAgICAgICAgICAgICAgVCksICAjICA2ICBneXJvX3JvdAogICAgICAgIHBhZDQobWFnX3JvdCwgICAgICAgICAgICAgICAgICAgIFQpLCAgIyAgNyAgbWFnX3JvdAogICAgICAgIHBhZDQoZGlmZihhY2MpLCAgICAgICAgICAgICAgICAgIFQpLCAgIyAgOCAgYWNjX2RlbHRhCiAgICAgICAgcGFkNChkaWZmKGd5cm8pLCAgICAgICAgICAgICAgICAgVCksICAjICA5ICBneXJvX2RlbHRhCiAgICAgICAgcGFkNChkaWZmKG1hZyksICAgICAgICAgICAgICAgICAgVCksICAjIDEwICBtYWdfZGVsdGEKICAgICAgICBwYWQ0KGRpZmYoYWNjX3JvdCksICAgICAgICAgICAgICBUKSwgICMgMTEgIGFjY19yb3RfZGVsdGEKICAgICAgICBwYWQ0KGRpZmYoZ3lyb19yb3QpLCAgICAgICAgICAgICBUKSwgICMgMTIgIGd5cm9fcm90X2RlbHRhCiAgICAgICAgcGFkNChkaWZmKG1hZ19yb3QpLCAgICAgICAgICAgICAgVCksICAjIDEzICBtYWdfcm90X2RlbHRhCiAgICAgICAgcGFkNChucC5jdW1zdW0oYWNjLCAgICAgIGF4aXM9MCksIFQpLCAjIDE0ICBhY2NfaW50ZWdyYWwKICAgICAgICBwYWQ0KG5wLmN1bXN1bShneXJvLCAgICAgYXhpcz0wKSwgVCksICMgMTUgIGd5cm9faW50ZWdyYWwKICAgICAgICBwYWQ0KG5wLmN1bXN1bShtYWcsICAgICAgYXhpcz0wKSwgVCksICMgMTYgIG1hZ19pbnRlZ3JhbAogICAgICAgIHBhZDQobnAuY3Vtc3VtKGFjY19yb3QsICBheGlzPTApLCBUKSwgIyAxNyAgYWNjX3JvdF9pbnRlZ3JhbAogICAgICAgIHBhZDQobnAuY3Vtc3VtKGd5cm9fcm90LCBheGlzPTApLCBUKSwgIyAxOCAgZ3lyb19yb3RfaW50ZWdyYWwKICAgICAgICBwYWQ0KG5wLmN1bXN1bShtYWdfcm90LCAgYXhpcz0wKSwgVCksICMgMTkgIG1hZ19yb3RfaW50ZWdyYWwKICAgICAgICBwYWQ0KGRpZmYobGluX2FjYyksICAgICAgICAgICAgICBUKSwgICMgMjAgIGxpbl9hY2NfZGVsdGEKICAgICAgICBwYWQ0KG5wLmN1bXN1bShsaW5fYWNjLCAgYXhpcz0wKSwgVCksICMgMjEgIGxpbl9hY2NfaW50ZWdyYWwKICAgICAgICBwYWQ0KHJvdCwgICAgICAgICAgICAgICAgICAgICAgICBUKSwgICMgMjIgIHJvdF9yYXcKICAgIF0sIGF4aXM9MCkgICAjIOKGkiAoMjIsIDQsIFQpCgoKZGVmIHByZXByb2Nlc3NfdXNlcih1aWQ6IHN0ciwgc2Vzc2lvbl9kaXI6IHN0cik6CiAgICAiIiIKICAgIFByb2Nlc3MgYWxsIHVubG9jayBldmVudHMgZm9yIG9uZSB1c2VyLgogICAgUmV0dXJucyAoZmVhdHVyZXMsIGNsdXN0ZXJfaWRzKSBvciAoTm9uZSwgTm9uZSkuCiAgICAgIGZlYXR1cmVzOiAgICAoTl92YWxpZCwgMjIsIDQsIFQpCiAgICAgIGNsdXN0ZXJfaWRzOiAoTl92YWxpZCwpICB2YWx1ZXMgMC01CiAgICAiIiIKICAgIG1lcmdlZCwgc2NyZWVuID0gbG9hZF9zZXNzaW9uKHNlc3Npb25fZGlyKQogICAgaWYgbWVyZ2VkIGlzIE5vbmUgb3Igc2NyZWVuIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUKCiAgICB1bmxvY2tfcm93cyA9IChzY3JlZW5bc2NyZWVuWyJldmVudCJdID09IEZMQUdfVVNFUl9QUkVTRU5UXQogICAgICAgICAgICAgICAgICAgLnNvcnRfdmFsdWVzKCJ0aW1lc3RhbXAiKSkKICAgIGZlYXRzLCBjbHVzdGVycywgbl9za2lwID0gW10sIFtdLCAwCiAgICBmb3IgdHJpYWxfaWR4LCAoXywgcm93KSBpbiBlbnVtZXJhdGUodW5sb2NrX3Jvd3MuaXRlcnJvd3MoKSk6CiAgICAgICAgdyA9IGV4dHJhY3Rfd2luZG93KG1lcmdlZCwgaW50KHJvd1sidGltZXN0YW1wIl0pKQogICAgICAgIGlmIHcgaXMgTm9uZToKICAgICAgICAgICAgbl9za2lwICs9IDE7IGNvbnRpbnVlCiAgICAgICAgZmVhdHMuYXBwZW5kKGNvbXB1dGVfZmVhdHVyZXModykpCiAgICAgICAgY2x1c3RlcnMuYXBwZW5kKHRyaWFsX2lkeCAvLyA1MCkgICAgICAgICMgNiBjbHVzdGVycyDDlyA1MCB0cmlhbHMKCiAgICBpZiBuX3NraXA6CiAgICAgICAgcHJpbnQoZiIgIHVzZXI9e3VpZH06IHNraXBwZWQge25fc2tpcH0gdHJpYWxzIikKICAgIGlmIG5vdCBmZWF0czoKICAgICAgICByZXR1cm4gTm9uZSwgTm9uZQogICAgcmV0dXJuIChucC5zdGFjayhmZWF0cywgYXhpcz0wKS5hc3R5cGUobnAuZmxvYXQzMiksCiAgICAgICAgICAgIG5wLmFycmF5KGNsdXN0ZXJzLCBkdHlwZT1ucC5pbnQ2NCkpCgoKZGVmIHJ1bl9vbl9kdW1teSgpOgogICAgZnJvbSB1dGlscy5kdW1teV9kYXRhIGltcG9ydCBsb2FkX3V2X2R1bW15CiAgICBYLCBjbCA9IGxvYWRfdXZfZHVtbXkoKQogICAgYXNzZXJ0IFguc2hhcGUgPT0gKDEwMSwgMzAwLCAyMiwgNCwgNTApLCBmIldyb25nIHNoYXBlOiB7WC5zaGFwZX0iCiAgICBwcmludChmIkR1bW15IFVWOiBYPXtYLnNoYXBlfSAgT0siKQoKICAgICMgVGVzdCBjb21wdXRlX2ZlYXR1cmVzIG9uIHN5bnRoZXRpYyB3aW5kb3cKICAgIFQgPSA1MDsgbnAucmFuZG9tLnNlZWQoMCkKICAgIGZha2Vfd2luZG93ID0gcGQuRGF0YUZyYW1lKHsKICAgICAgICAiYWNjX1giOiAgbnAucmFuZG9tLnJhbmRuKFQpLCAiYWNjX1kiOiAgbnAucmFuZG9tLnJhbmRuKFQpLAogICAgICAgICJhY2NfWiI6ICBucC5yYW5kb20ucmFuZG4oVCkgKyA5LjgsCiAgICAgICAgImdyYXZfWCI6IG5wLnplcm9zKFQpLCAiZ3Jhdl9ZIjogbnAuemVyb3MoVCksCiAgICAgICAgImdyYXZfWiI6IG5wLmZ1bGwoVCwgOS44KSwKICAgICAgICAiZ3lyb19YIjogbnAucmFuZG9tLnJhbmRuKFQpKjAuMSwgImd5cm9fWSI6IG5wLnJhbmRvbS5yYW5kbihUKSowLjEsCiAgICAgICAgImd5cm9fWiI6IG5wLnJhbmRvbS5yYW5kbihUKSowLjEsCiAgICAgICAgImxpbl9YIjogIG5wLnJhbmRvbS5yYW5kbihUKSowLjEsICJsaW5fWSI6ICBucC5yYW5kb20ucmFuZG4oVCkqMC4xLAogICAgICAgICJsaW5fWiI6ICBucC5yYW5kb20ucmFuZG4oVCkqMC4xLAogICAgICAgICJtYWdfWCI6ICBucC5yYW5kb20ucmFuZG4oVCkqMTAsICAibWFnX1kiOiAgbnAucmFuZG9tLnJhbmRuKFQpKjEwLAogICAgICAgICJtYWdfWiI6ICBucC5yYW5kb20ucmFuZG4oVCkqMTAsCiAgICAgICAgInJvdF9YIjogIG5wLnJhbmRvbS5yYW5kbihUKSowLjEsICJyb3RfWSI6ICBucC5yYW5kb20ucmFuZG4oVCkqMC4xLAogICAgICAgICJyb3RfWiI6ICBucC5yYW5kb20ucmFuZG4oVCkqMC4xLAogICAgfSkKICAgIGZlYXRzID0gY29tcHV0ZV9mZWF0dXJlcyhmYWtlX3dpbmRvdykKICAgIGFzc2VydCBmZWF0cy5zaGFwZSA9PSAoMjIsIDQsIFQpLCBmIkV4cGVjdGVkICgyMiw0LHtUfSksIGdvdCB7ZmVhdHMuc2hhcGV9IgogICAgYXNzZXJ0IG5wLmFsbChmZWF0c1s6LCAzLCA6XSA9PSAwLjApLCAiNHRoIGNoYW5uZWwgbXVzdCBiZSB6ZXJvIChwYWRkaW5nKSIKICAgIGFzc2VydCBucC5hbnkoZmVhdHNbOiwgMCwgOl0gIT0gMC4wKSwgIkZlYXR1cmUgZGF0YSBtdXN0IGJlIG5vbi16ZXJvIgogICAgcHJpbnQoZiIgIGNvbXB1dGVfZmVhdHVyZXM6IHtmZWF0cy5zaGFwZX0gIE9LIikKICAgIHByaW50KGYiICA0dGggY2hhbm5lbCBwYWRkaW5nOiBjb25maXJtZWQgYWxsLXplcm8iKQogICAgcHJpbnQoIkR1bW15IFVWIHRlc3QgcGFzc2VkLiIpCgoKZGVmIG1haW4odXNlX2R1bW15PUZhbHNlKToKICAgIGlmIHVzZV9kdW1teToKICAgICAgICBydW5fb25fZHVtbXkoKTsgcmV0dXJuCiAgICBvcy5tYWtlZGlycyhQUk9DRVNTRURfRElSLCBleGlzdF9vaz1UcnVlKQogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKFVWX0lOUFVUX0RJUik6CiAgICAgICAgcHJpbnQoZiJVViBpbnB1dCBub3QgZm91bmQ6IHtVVl9JTlBVVF9ESVJ9Iik7IHJldHVybgogICAgdXNlcl9kaXJzID0gc29ydGVkKAogICAgICAgIGQgZm9yIGQgaW4gb3MubGlzdGRpcihVVl9JTlBVVF9ESVIpCiAgICAgICAgaWYgb3MucGF0aC5pc2Rpcihvcy5wYXRoLmpvaW4oVVZfSU5QVVRfRElSLCBkKSkpCiAgICBwcmludChmIkZvdW5kIHtsZW4odXNlcl9kaXJzKX0gdXNlcnMgaW4gVVYgZGF0YXNldC4iKQogICAgZm9yIHVpZCBpbiB1c2VyX2RpcnM6CiAgICAgICAgIyBTdHJ1Y3R1cmU6IHt1c2VyX2lkfS9zMjAve3VzZXJfaWR9XzIwMDAwLwogICAgICAgIGJhc2UgICAgICA9IG9zLnBhdGguam9pbihVVl9JTlBVVF9ESVIsIHVpZCwgInMyMCIpCiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGJhc2UpOgogICAgICAgICAgICBwcmludChmIiAgdXNlcj17dWlkfTogbm8gczIwIGRpciwgc2tpcHBpbmciKTsgY29udGludWUKICAgICAgICBzZXNzaW9ucyAgPSBzb3J0ZWQob3MubGlzdGRpcihiYXNlKSkKICAgICAgICBpZiBub3Qgc2Vzc2lvbnM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgc2RpciAgICAgID0gb3MucGF0aC5qb2luKGJhc2UsIHNlc3Npb25zWzBdKQogICAgICAgIGZlYXRzLCBjbCA9IHByZXByb2Nlc3NfdXNlcih1aWQsIHNkaXIpCiAgICAgICAgaWYgZmVhdHMgaXMgTm9uZToKICAgICAgICAgICAgcHJpbnQoZiIgIHVzZXI9e3VpZH06IG5vIHZhbGlkIHRyaWFscyIpOyBjb250aW51ZQogICAgICAgIG5wLnNhdmV6KG9zLnBhdGguam9pbihQUk9DRVNTRURfRElSLCBmInt1aWR9Lm5weiIpLAogICAgICAgICAgICAgICAgIGZlYXR1cmVzPWZlYXRzLCBjbHVzdGVyX2lkcz1jbCwgdXNlcl9pZD11aWQpCiAgICAgICAgcHJpbnQoZiIgIHVzZXI9e3VpZH06IHtmZWF0cy5zaGFwZX0iKQogICAgcHJpbnQoIlVWIHByZXByb2Nlc3NpbmcgZG9uZS4iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXVzZS1kdW1teSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgbWFpbih1c2VfZHVtbXk9YXJncy51c2VfZHVtbXkpCg==", "models/__init__.py": "IyBtb2RlbHMgcGFja2FnZQo=", "models/losses.py": "IiIiCkxvc3MgZnVuY3Rpb25zIGZvciBNb3Rpb24gSUQgVVYgc3RhZ2UuClBhcGVyIFNlY3Rpb24gNi40OiBMX3RvdGFsID0gTENFICsgYWxwaGFfVE0gKiBMVE0gKyBMU0MKICBMQ0UgPSBjcm9zcy1lbnRyb3B5IChjbGFzc2lmaWVyIGhlYWQpCiAgTFRNID0gVHJpcGxldCBNYXJnaW4gTG9zcywgc2VtaS1oYXJkIG1pbmluZyAoU2Nocm9mZiBldCBhbC4gMjAxNSkKICBMU0MgPSBTdXBlcnZpc2VkIENvbnRyYXN0aXZlIExvc3MgKEtob3NsYSBldCBhbC4gMjAyMCkKClJ1bjogcHl0aG9uIG1vZGVscy9sb3NzZXMucHkKRXhwZWN0ZWQ6IEFsbCBsb3NzIHVuaXQgdGVzdHMgcGFzc2VkLgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmNsYXNzIENyb3NzRW50cm9weUxvc3Mobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLl9jZSA9IG5uLkNyb3NzRW50cm9weUxvc3MoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgbGFiZWxzKToKICAgICAgICByZXR1cm4gc2VsZi5fY2UobG9naXRzLCBsYWJlbHMpCgoKY2xhc3MgVHJpcGxldE1hcmdpbkxvc3Mobm4uTW9kdWxlKToKICAgICIiIgogICAgU2VtaS1oYXJkIG5lZ2F0aXZlIG1pbmluZy4KICAgIEZvciBlYWNoIGFuY2hvcjogZmluZCBuZWdhdGl2ZXMgZmFydGhlciB0aGFuIHRoZSBwb3NpdGl2ZQogICAgYnV0IHdpdGhpbiBtYXJnaW4uIEZhbGxzIGJhY2sgdG8gaGFyZGVzdCBuZWdhdGl2ZSBpZiBub25lIGZvdW5kLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgbWFyZ2luOiBmbG9hdCA9IDEuMCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5tYXJnaW4gPSBtYXJnaW4KCiAgICBkZWYgZm9yd2FyZChzZWxmLCBlbWJlZGRpbmdzLCBsYWJlbHMpOgogICAgICAgIGRldmljZSA9IGVtYmVkZGluZ3MuZGV2aWNlCiAgICAgICAgQiAgICAgID0gZW1iZWRkaW5ncy5zaXplKDApCiAgICAgICAgZGlzdCAgID0gKChlbWJlZGRpbmdzLnVuc3F1ZWV6ZSgwKSAtCiAgICAgICAgICAgICAgICAgICBlbWJlZGRpbmdzLnVuc3F1ZWV6ZSgxKSkgKiogMikuc3VtKDIpICAgIyAoQiwgQikKICAgICAgICBsZXEgICAgPSBsYWJlbHMudW5zcXVlZXplKDApID09IGxhYmVscy51bnNxdWVlemUoMSkKICAgICAgICBleWUgICAgPSB0b3JjaC5leWUoQiwgZHR5cGU9dG9yY2guYm9vbCwgZGV2aWNlPWRldmljZSkKCiAgICAgICAgbG9zc2VzID0gW10KICAgICAgICBmb3IgaSBpbiByYW5nZShCKToKICAgICAgICAgICAgcG0gPSBsZXFbaV0gJiB+ZXllW2ldICAgIyB2YWxpZCBwb3NpdGl2ZXMKICAgICAgICAgICAgbm0gPSB+bGVxW2ldICAgICAgICAgICAgIyB2YWxpZCBuZWdhdGl2ZXMKICAgICAgICAgICAgaWYgbm90IHBtLmFueSgpIG9yIG5vdCBubS5hbnkoKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGRfYXAgID0gZGlzdFtpXVtwbV0ubWF4KCkKICAgICAgICAgICAgZF9uZWcgPSBkaXN0W2ldW25tXQogICAgICAgICAgICBzZW1pICA9IGRfbmVnWyhkX25lZyA+IGRfYXApICYgKGRfbmVnIDwgZF9hcCArIHNlbGYubWFyZ2luKV0KICAgICAgICAgICAgZF9hbiAgPSBzZW1pLm1pbigpIGlmIHNlbWkubnVtZWwoKSA+IDAgZWxzZSBkX25lZy5taW4oKQogICAgICAgICAgICBsb3NzZXMuYXBwZW5kKEYucmVsdShkX2FwIC0gZF9hbiArIHNlbGYubWFyZ2luKSkKCiAgICAgICAgaWYgbm90IGxvc3NlczoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcigwLjAsIHJlcXVpcmVzX2dyYWQ9VHJ1ZSwgZGV2aWNlPWRldmljZSkKICAgICAgICByZXR1cm4gdG9yY2guc3RhY2sobG9zc2VzKS5tZWFuKCkKCgpjbGFzcyBTdXBlcnZpc2VkQ29udHJhc3RpdmVMb3NzKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIEtob3NsYSBldCBhbC4gMjAyMCDigJQgY2l0ZWQgaW4gcGFwZXIgU2VjdGlvbiA2LjQuCiAgICBFeHBlY3RzIHByb2pfZW1iZWRkaW5ncyBvZiBzaGFwZSAoMkIsIEQpOiB0d28gYXVnbWVudGVkIHZpZXdzCiAgICBjb25jYXRlbmF0ZWQgYWxvbmcgdGhlIGJhdGNoIGRpbWVuc2lvbi4gbGFiZWxzIHNoYXBlIChCLCkuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjA3KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnRlbXBlcmF0dXJlID0gdGVtcGVyYXR1cmUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBwcm9qX2VtYmVkZGluZ3MsIGxhYmVscyk6CiAgICAgICAgZGV2aWNlICAgICA9IHByb2pfZW1iZWRkaW5ncy5kZXZpY2UKICAgICAgICBOICAgICAgICAgID0gcHJval9lbWJlZGRpbmdzLnNpemUoMCkKICAgICAgICBCICAgICAgICAgID0gbGFiZWxzLnNpemUoMCkKICAgICAgICBsYWJlbHNfcmVwID0gbGFiZWxzLnJlcGVhdCgyKSBpZiBOID09IDIgKiBCIGVsc2UgbGFiZWxzCgogICAgICAgIHogICAgICAgID0gRi5ub3JtYWxpemUocHJval9lbWJlZGRpbmdzLCBkaW09MSkKICAgICAgICBzaW0gICAgICA9IHRvcmNoLm1tKHosIHouVCkgLyBzZWxmLnRlbXBlcmF0dXJlCiAgICAgICAgZXllICAgICAgPSB0b3JjaC5leWUoTiwgZHR5cGU9dG9yY2guYm9vbCwgZGV2aWNlPWRldmljZSkKICAgICAgICBwb3NfbWFzayA9IChsYWJlbHNfcmVwLnVuc3F1ZWV6ZSgwKSA9PQogICAgICAgICAgICAgICAgICAgIGxhYmVsc19yZXAudW5zcXVlZXplKDEpKSAmIH5leWUKCiAgICAgICAgIyBOdW1lcmljYWwgc3RhYmlsaXR5CiAgICAgICAgc2ltICAgICAgPSBzaW0gLSBzaW0ubWF4KGRpbT0xLCBrZWVwZGltPVRydWUpLnZhbHVlcy5kZXRhY2goKQogICAgICAgIGV4cF9zaW0gID0gdG9yY2guZXhwKHNpbSkKICAgICAgICBkZW5vbSAgICA9IGV4cF9zaW0ubWFza2VkX2ZpbGwoZXllLCAwKS5zdW0oMSwga2VlcGRpbT1UcnVlKQogICAgICAgIGxvZ19wcm9iID0gc2ltIC0gdG9yY2gubG9nKGRlbm9tICsgMWUtOCkKCiAgICAgICAgbl9wb3MgPSBwb3NfbWFzay5zdW0oMSkuZmxvYXQoKQogICAgICAgIHZhbGlkID0gbl9wb3MgPiAwCiAgICAgICAgbG9zcyAgPSAtKGxvZ19wcm9iICogcG9zX21hc2suZmxvYXQoKSkuc3VtKDEpCiAgICAgICAgcmV0dXJuIChsb3NzW3ZhbGlkXSAvIG5fcG9zW3ZhbGlkXSkubWVhbigpCgoKY2xhc3MgVG90YWxMb3NzKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFBhcGVyIFNlY3Rpb24gNi40OiBMX3RvdGFsID0gTENFICsgYWxwaGFfVE0gKiBMVE0gKyBMU0MKICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFscGhhX3RtOiBmbG9hdCA9IDEuMCwgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wNyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5hbHBoYV90bSA9IGFscGhhX3RtCiAgICAgICAgc2VsZi5jZSAgICAgICA9IENyb3NzRW50cm9weUxvc3MoKQogICAgICAgIHNlbGYudG0gICAgICAgPSBUcmlwbGV0TWFyZ2luTG9zcyhtYXJnaW49MS4wKQogICAgICAgIHNlbGYuc2MgICAgICAgPSBTdXBlcnZpc2VkQ29udHJhc3RpdmVMb3NzKHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgcHJval9lbWJlZHMsIGxhYmVscyk6CiAgICAgICAgIiIiCiAgICAgICAgbG9naXRzOiAgICAgIChCLCBuX2NsYXNzZXMpCiAgICAgICAgcHJval9lbWJlZHM6ICgyQiwgRCkgIOKAlCB0d28gYXVnbWVudGVkIHZpZXdzIGNvbmNhdGVuYXRlZAogICAgICAgIGxhYmVsczogICAgICAoQiwpCiAgICAgICAgUmV0dXJuczogKHNjYWxhciB0b3RhbCBsb3NzLCBkaWN0IG9mIGNvbXBvbmVudCB2YWx1ZXMpCiAgICAgICAgIiIiCiAgICAgICAgbGNlICAgPSBzZWxmLmNlKGxvZ2l0cywgbGFiZWxzKQogICAgICAgIGx0bSAgID0gc2VsZi50bSgKICAgICAgICAgICAgRi5ub3JtYWxpemUocHJval9lbWJlZHNbOmxhYmVscy5zaXplKDApXSwgZGltPTEpLCBsYWJlbHMpCiAgICAgICAgbHNjICAgPSBzZWxmLnNjKHByb2pfZW1iZWRzLCBsYWJlbHMpCiAgICAgICAgdG90YWwgPSBsY2UgKyBzZWxmLmFscGhhX3RtICogbHRtICsgbHNjCiAgICAgICAgcmV0dXJuIHRvdGFsLCB7CiAgICAgICAgICAgICJsY2UiOiBsY2UuaXRlbSgpLAogICAgICAgICAgICAibHRtIjogbHRtLml0ZW0oKSwKICAgICAgICAgICAgImxzYyI6IGxzYy5pdGVtKCl9CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDApCiAgICBCLCBELCBuX2NscyA9IDgsIDY0LCAxMAogICAgZW1iICAgID0gRi5ub3JtYWxpemUodG9yY2gucmFuZG4oMiAqIEIsIEQpLCBkaW09MSkKICAgIGxvZ2l0cyA9IHRvcmNoLnJhbmRuKEIsIG5fY2xzKQogICAgbGFiZWxzID0gdG9yY2gudGVuc29yKFswLCAwLCAxLCAxLCAyLCAyLCAzLCAzXSkKCiAgICBsb3NzX2ZuICAgICAgID0gVG90YWxMb3NzKGFscGhhX3RtPTEuMCwgdGVtcGVyYXR1cmU9MC4wNykKICAgIGxvc3MsIHBhcnRzICAgPSBsb3NzX2ZuKGxvZ2l0cywgZW1iLCBsYWJlbHMpCgogICAgcHJpbnQoZiJUb3RhbCBsb3NzOiB7bG9zcy5pdGVtKCk6LjRmfSIpCiAgICBmb3IgaywgdiBpbiBwYXJ0cy5pdGVtcygpOgogICAgICAgIHByaW50KGYiICB7a306IHt2Oi40Zn0iKQogICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGxvc3MpIGFuZCBsb3NzLml0ZW0oKSA+IDAKICAgIHByaW50KCJBbGwgbG9zcyB1bml0IHRlc3RzIHBhc3NlZC4iKQo=", "models/uv_model.py": "IiIiClVWIG1vZGVsOiAyMiBwYXJhbGxlbCAxRC1DTk4gYnJhbmNoZXMgKyBkdWFsIGhlYWQuCk1QSSBtb2RlbDogc2ltcGxlIDFELUNOTiBiaW5hcnkgY2xhc3NpZmllci4KUGFwZXI6IFNlY3Rpb25zIDUuMSBhbmQgNi40IG9mIGFyWGl2OjIzMDIuMDE3NTEKClNpbmdsZSBHUFUgb25seS4gTm8gRGF0YVBhcmFsbGVsLgoKUnVuOiBweXRob24gbW9kZWxzL3V2X21vZGVsLnB5CkV4cGVjdGVkOgogIFVWIHBhcmFtczogfjgsMzc3LDU0NwogIGxvZ2l0czogdG9yY2guU2l6ZShbNCwgNjBdKQogIHByb2o6ICAgdG9yY2guU2l6ZShbNCwgNjRdKQogIHNpYW1lc2Ugbm9ybXM6IGFsbCB+MS4wCiAgTVBJIG91dHB1dDogdG9yY2guU2l6ZShbOCwgMl0pCiAgQWxsIG1vZGVsIHRlc3RzIHBhc3NlZC4KIiIiCgppbXBvcnQgc3lzLCBvcwppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkKZnJvbSBjb25maWdzLmNvbmZpZyBpbXBvcnQgY2ZnCgoKY2xhc3MgVVZCcmFuY2gobm4uTW9kdWxlKToKICAgICIiIgogICAgT25lIG9mIDIyIGlkZW50aWNhbCBwYXJhbGxlbCBicmFuY2hlcy4KICAgIElucHV0OiAgKGJhdGNoLCA0LCBUKQogICAgT3V0cHV0OiAoYmF0Y2gsIDI1NikKICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQgPSA0KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYxID0gbm4uQ29udjFkKGluX2NoYW5uZWxzLCAzMiwgMywgcGFkZGluZz0xKQogICAgICAgIHNlbGYuYm4xICAgPSBubi5CYXRjaE5vcm0xZCgzMikKICAgICAgICBzZWxmLmNvbnYyID0gbm4uQ29udjFkKDMyLCA2NCwgMywgcGFkZGluZz0xKQogICAgICAgIHNlbGYuYm4yICAgPSBubi5CYXRjaE5vcm0xZCg2NCkKICAgICAgICBzZWxmLmNvbnYzID0gbm4uQ29udjFkKDY0LCAxMjgsIDMsIHBhZGRpbmc9MSkKICAgICAgICBzZWxmLmJuMyAgID0gbm4uQmF0Y2hOb3JtMWQoMTI4KQogICAgICAgIHNlbGYucG9vbCAgPSBubi5BZGFwdGl2ZUF2Z1Bvb2wxZCg4KQogICAgICAgIHNlbGYuZmMgICAgPSBubi5MaW5lYXIoMTI4ICogOCwgMjU2KQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHggPSBGLnJlbHUoc2VsZi5ibjEoc2VsZi5jb252MSh4KSkpCiAgICAgICAgeCA9IEYucmVsdShzZWxmLmJuMihzZWxmLmNvbnYyKHgpKSkKICAgICAgICB4ID0gRi5yZWx1KHNlbGYuYm4zKHNlbGYuY29udjMoeCkpKQogICAgICAgIHJldHVybiBzZWxmLmZjKHNlbGYucG9vbCh4KS5mbGF0dGVuKDEpKQoKCmNsYXNzIFVWTW9kZWwobm4uTW9kdWxlKToKICAgICIiIgogICAgUGFwZXIgU2VjdGlvbiA2LjQgYXJjaGl0ZWN0dXJlOgogICAgICAyMiBwYXJhbGxlbCBicmFuY2hlcyDihpIgY29uY2F0ZW5hdGUg4oaSIGR1YWwgaGVhZDoKICAgICAgICBIZWFkIEE6IExpbmVhciDihpIgbl9jbGFzc2VzICAoY3Jvc3MtZW50cm9weSwgY2xhc3NpZmllcikKICAgICAgICBIZWFkIEI6IExpbmVhciDihpIgTDItbm9ybSDihpIgTUxQICAoU2lhbWVzZSArIFN1cENvbikKICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG5fY2xhc3NlczogaW50LCBuX2ZlYXR1cmVzOiBpbnQgPSAyMik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5uX2ZlYXR1cmVzID0gbl9mZWF0dXJlcwogICAgICAgIHNlbGYuZW1iZWRfZGltICA9IG5fZmVhdHVyZXMgKiAyNTYgICAjIDIyICogMjU2ID0gNTYzMgoKICAgICAgICBzZWxmLmJyYW5jaGVzICAgICAgPSBubi5Nb2R1bGVMaXN0KAogICAgICAgICAgICBbVVZCcmFuY2goaW5fY2hhbm5lbHM9NCkgZm9yIF8gaW4gcmFuZ2Uobl9mZWF0dXJlcyldKQogICAgICAgIHNlbGYuaGVhZF9hICAgICAgICA9IG5uLkxpbmVhcihzZWxmLmVtYmVkX2RpbSwgbl9jbGFzc2VzKQogICAgICAgIHNlbGYuc2lhbWVzZV9wcm9qICA9IG5uLkxpbmVhcihzZWxmLmVtYmVkX2RpbSwgMjU2KQogICAgICAgIHNlbGYuaGVhZF9iICAgICAgICA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcigyNTYsIDEyOCksIG5uLlJlTFUoKSwgbm4uTGluZWFyKDEyOCwgNjQpKQoKICAgIGRlZiBfYXVnbWVudChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiIKICAgICAgICBQYXBlciBTZWN0aW9uIDYuMjogcmFuZG9tbHkgY3V0IDEuNS1zZWNvbmQgc2VyaWVzIHRvIDEgc2Vjb25kLAogICAgICAgIHRoZW4gYWRkIHJhbmRvbSBHYXVzc2lhbiBub2lzZS4KICAgICAgICAiIiIKICAgICAgICBUX3QgPSBpbnQoY2ZnLnV2X3dpbmRvd19zZWMgKiBjZmcudXZfc2FtcGxpbmdfcmF0ZSkgICMgNTAKICAgICAgICBUICAgPSB4LnNpemUoLTEpCiAgICAgICAgaWYgVCA+IFRfdDoKICAgICAgICAgICAgc3RhcnQgPSB0b3JjaC5yYW5kaW50KDAsIFQgLSBUX3QgKyAxLCAoMSwpKS5pdGVtKCkKICAgICAgICAgICAgeCAgICAgPSB4Wy4uLiwgc3RhcnQgOiBzdGFydCArIFRfdF0KICAgICAgICByZXR1cm4geCArIHRvcmNoLnJhbmRuX2xpa2UoeCkgKiAwLjAxCgogICAgZGVmIGV4dHJhY3RfZW1iZWRkaW5nKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIng6IChCLCAyMiwgNCwgVCkg4oaSIChCLCA1NjMyKSIiIgogICAgICAgIHJldHVybiB0b3JjaC5jYXQoCiAgICAgICAgICAgIFtzZWxmLmJyYW5jaGVzW2ldKHhbOiwgaSwgOiwgOl0pCiAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShzZWxmLm5fZmVhdHVyZXMpXSwgZGltPTEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yLCBhdWdtZW50OiBib29sID0gRmFsc2UpOgogICAgICAgICIiIgogICAgICAgIHg6IChCLCAyMiwgNCwgVCkKICAgICAgICBSZXR1cm5zOiBsb2dpdHMgKEIsIG5fY2xhc3NlcyksICBwcm9qIChCLCA2NCkKICAgICAgICAiIiIKICAgICAgICBpZiBhdWdtZW50OgogICAgICAgICAgICB4ID0gc2VsZi5fYXVnbWVudCh4KQogICAgICAgIGVtYiAgICAgPSBzZWxmLmV4dHJhY3RfZW1iZWRkaW5nKHgpCiAgICAgICAgbG9naXRzICA9IHNlbGYuaGVhZF9hKGVtYikKICAgICAgICBzaWFtZXNlID0gRi5ub3JtYWxpemUoc2VsZi5zaWFtZXNlX3Byb2ooZW1iKSwgZGltPTEpCiAgICAgICAgcmV0dXJuIGxvZ2l0cywgc2VsZi5oZWFkX2Ioc2lhbWVzZSkKCiAgICBkZWYgZ2V0X3NpYW1lc2VfZW1iZWQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiSW5mZXJlbmNlIG9ubHkuIFJldHVybnMgTDItbm9ybWFsaXNlZCAoQiwgMjU2KSBlbWJlZGRpbmcuIiIiCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGVtYiA9IHNlbGYuZXh0cmFjdF9lbWJlZGRpbmcoeCkKICAgICAgICAgICAgcmV0dXJuIEYubm9ybWFsaXplKHNlbGYuc2lhbWVzZV9wcm9qKGVtYiksIGRpbT0xKQoKCmNsYXNzIE1QSU1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFBhcGVyIFNlY3Rpb24gNS4xOiBDTk4gd2l0aCBwb2ludHdpc2UgY29udm9sdXRpb25zLAogICAgY3Jvc3MtZW50cm9weSBsb3NzLCAyIG91dHB1dCBjbGFzc2VzICh1bmxvY2sgLyBuby11bmxvY2spLgogICAgSW5wdXQ6IChCLCBDLCBUKSDihpIgKEIsIDIpCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuX2NoYW5uZWxzOiBpbnQgPSAxOCwgbl9jbGFzc2VzOiBpbnQgPSAyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5ldCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYxZChuX2NoYW5uZWxzLCA2NCwgNSwgcGFkZGluZz0yKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMWQoNjQpLCBubi5SZUxVKCksCiAgICAgICAgICAgIG5uLkNvbnYxZCg2NCwgMTI4LCA1LCBwYWRkaW5nPTIpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0xZCgxMjgpLCBubi5SZUxVKCksCiAgICAgICAgICAgIG5uLkNvbnYxZCgxMjgsIDI1NiwgMywgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMWQoMjU2KSwgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5BZGFwdGl2ZUF2Z1Bvb2wxZCg4KSkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5GbGF0dGVuKCksCiAgICAgICAgICAgIG5uLkxpbmVhcigyNTYgKiA4LCAyNTYpLCBubi5SZUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcigyNTYsIG5fY2xhc3NlcykpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3NpZmllcihzZWxmLm5ldCh4KSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdG9yY2gubWFudWFsX3NlZWQoNDIpCgogICAgcHJpbnQoIlVWIE1vZGVsIHRlc3QiKQogICAgbW9kZWwgPSBVVk1vZGVsKG5fY2xhc3Nlcz02MCkKICAgIHByaW50KGYiICBVViBwYXJhbXM6IHtzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSk6LH0iKQoKICAgIHggPSB0b3JjaC5yYW5kbig0LCAyMiwgNCwgNTApCiAgICBsb2dpdHMsIHByb2ogPSBtb2RlbCh4KQogICAgYXNzZXJ0IGxvZ2l0cy5zaGFwZSA9PSAoNCwgNjApLCBmIkdvdCB7bG9naXRzLnNoYXBlfSIKICAgIGFzc2VydCBwcm9qLnNoYXBlICAgPT0gKDQsIDY0KSwgZiJHb3Qge3Byb2ouc2hhcGV9IgogICAgcHJpbnQoZiIgIGxvZ2l0czoge2xvZ2l0cy5zaGFwZX0iKQogICAgcHJpbnQoZiIgIHByb2o6ICAge3Byb2ouc2hhcGV9IikKCiAgICB4NzUgPSB0b3JjaC5yYW5kbig0LCAyMiwgNCwgNzUpCiAgICBsMiwgXyA9IG1vZGVsKHg3NSwgYXVnbWVudD1UcnVlKQogICAgYXNzZXJ0IGwyLnNoYXBlID09ICg0LCA2MCkKICAgIHByaW50KGYiICBhdWdtZW50IFQ9NzUgaW5wdXQg4oaSIHtsMi5zaGFwZX0gIE9LIikKCiAgICBzaWFtICA9IG1vZGVsLmdldF9zaWFtZXNlX2VtYmVkKHgpCiAgICBub3JtcyA9IHNpYW0ubm9ybShkaW09MSkKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShub3JtcywgdG9yY2gub25lcyg0KSwgYXRvbD0xZS01KQogICAgcHJpbnQoZiIgIHNpYW1lc2Ugbm9ybXM6IHtub3Jtcy50b2xpc3QoKX0gIGFsbCB+MS4wIikKCiAgICBwcmludCgiXG5NUEkgTW9kZWwgdGVzdCIpCiAgICBtcGkgPSBNUElNb2RlbChuX2NoYW5uZWxzPTE4LCBuX2NsYXNzZXM9MikKICAgIG91dCA9IG1waSh0b3JjaC5yYW5kbig4LCAxOCwgMTUwKSkKICAgIGFzc2VydCBvdXQuc2hhcGUgPT0gKDgsIDIpCiAgICBwcmludChmIiAgTVBJIG91dHB1dDoge291dC5zaGFwZX0iKQoKICAgIHByaW50KCJcbkFsbCBtb2RlbCB0ZXN0cyBwYXNzZWQuIikK", "training/__init__.py": "IyB0cmFpbmluZyBwYWNrYWdlCg==", "training/train_mpi.py": "IiIiCk1QSSB0cmFpbmluZy4gRnVsbHkgcmVzdW1hYmxlLgpQYXBlcjogU2VjdGlvbiA1LjEgb2YgYXJYaXY6MjMwMi4wMTc1MQpTaW5nbGUgR1BVIG9ubHkg4oCUIG5vIERhdGFQYXJhbGxlbC4KUnVuOiBweXRob24gdHJhaW5pbmcvdHJhaW5fbXBpLnB5IC0tdXNlLWR1bW15CiIiIgoKaW1wb3J0IG9zLCBzeXMsIGFyZ3BhcnNlCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBUZW5zb3JEYXRhc2V0LCBEYXRhTG9hZGVyCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IFN0cmF0aWZpZWRTaHVmZmxlU3BsaXQKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUsIGFjY3VyYWN5X3Njb3JlCgpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpCmZyb20gY29uZmlncy5jb25maWcgaW1wb3J0IGNmZwpmcm9tIG1vZGVscy51dl9tb2RlbCBpbXBvcnQgTVBJTW9kZWwKClBST0NFU1NFRF9ESVIgPSAiZGF0YS9tcGkvcHJvY2Vzc2VkIgpDS1BUX0RJUiAgICAgID0gIm1vZGVscy9jaGVja3BvaW50cyIKUkVTVUxUU19ESVIgICA9ICJldmFsdWF0aW9uIgpSRVNVTUVfTE9HICAgID0gb3MucGF0aC5qb2luKFJFU1VMVFNfRElSLCAibXBpX2NvbXBsZXRlZC5jc3YiKQoKb3MubWFrZWRpcnMoQ0tQVF9ESVIsICAgZXhpc3Rfb2s9VHJ1ZSkKb3MubWFrZWRpcnMoUkVTVUxUU19ESVIsIGV4aXN0X29rPVRydWUpCgpkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKCgpkZWYgbG9hZF9jb21wbGV0ZWQoKSAtPiBzZXQ6CiAgICBpZiBvcy5wYXRoLmV4aXN0cyhSRVNVTUVfTE9HKToKICAgICAgICByZXR1cm4gc2V0KG1hcCh0dXBsZSwKICAgICAgICAgICAgcGQucmVhZF9jc3YoUkVTVU1FX0xPRylbWyJ1c2VyX2lkIiwiZGV2aWNlX2lkIiwic2VlZCJdXQogICAgICAgICAgICAgIC52YWx1ZXMudG9saXN0KCkpKQogICAgcmV0dXJuIHNldCgpCgoKZGVmIG1hcmtfY29tcGxldGVkKHVpZCwgZGlkLCBzZWVkKToKICAgIHJvdyA9IHBkLkRhdGFGcmFtZShbeyJ1c2VyX2lkIjogdWlkLCAiZGV2aWNlX2lkIjogZGlkLCAic2VlZCI6IHNlZWR9XSkKICAgIHJvdy50b19jc3YoUkVTVU1FX0xPRywgbW9kZT0iYSIsCiAgICAgICAgICAgICAgIGhlYWRlcj1ub3Qgb3MucGF0aC5leGlzdHMoUkVTVU1FX0xPRyksIGluZGV4PUZhbHNlKQoKCmRlZiB0cmFpbl9vbmVfcGFpcihYLCB5LCB1aWQsIGRpZCwgc2VlZDogaW50KSAtPiB0dXBsZToKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpOyBucC5yYW5kb20uc2VlZChzZWVkKQoKICAgICMgU3RyYXRpZmllZCA3MCAvIDE1IC8gMTUgc3BsaXQgYnkgc2FtcGxlCiAgICBzc3MxID0gU3RyYXRpZmllZFNodWZmbGVTcGxpdCgxLCB0ZXN0X3NpemU9MC4zMCwgcmFuZG9tX3N0YXRlPXNlZWQpCiAgICBpZHhfdHIsIGlkeF90bXAgPSBuZXh0KHNzczEuc3BsaXQoWCwgeSkpCiAgICBzc3MyID0gU3RyYXRpZmllZFNodWZmbGVTcGxpdCgxLCB0ZXN0X3NpemU9MC41MCwgcmFuZG9tX3N0YXRlPXNlZWQpCiAgICBpZHhfdiwgaWR4X3RlICAgPSBuZXh0KHNzczIuc3BsaXQoWFtpZHhfdG1wXSwgeVtpZHhfdG1wXSkpCiAgICBpZHhfdiAgPSBpZHhfdG1wW2lkeF92XQogICAgaWR4X3RlID0gaWR4X3RtcFtpZHhfdGVdCgogICAgbW9kZWwgPSBNUElNb2RlbChuX2NoYW5uZWxzPVguc2hhcGVbMV0sIG5fY2xhc3Nlcz0yKS50byhkZXZpY2UpCiAgICBvcHQgICA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj1jZmcuYmFzZWxpbmVfbHIpCiAgICBjcml0ICA9IG5uLkNyb3NzRW50cm9weUxvc3MoKQoKICAgIGRlZiBtYWtlX2xvYWRlcihYYSwgeWEsIHNodWZmbGU9VHJ1ZSk6CiAgICAgICAgcmV0dXJuIERhdGFMb2FkZXIoCiAgICAgICAgICAgIFRlbnNvckRhdGFzZXQodG9yY2gudGVuc29yKFhhKSwKICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC50ZW5zb3IoeWEsIGR0eXBlPXRvcmNoLmxvbmcpKSwKICAgICAgICAgICAgYmF0Y2hfc2l6ZT0zMiwgc2h1ZmZsZT1zaHVmZmxlKQoKICAgIHRyX2xvID0gbWFrZV9sb2FkZXIoWFtpZHhfdHJdLCB5W2lkeF90cl0pCiAgICB2YV9sbyA9IG1ha2VfbG9hZGVyKFhbaWR4X3ZdLCAgeVtpZHhfdl0sICBzaHVmZmxlPUZhbHNlKQogICAgdGVfbG8gPSBtYWtlX2xvYWRlcihYW2lkeF90ZV0sIHlbaWR4X3RlXSwgc2h1ZmZsZT1GYWxzZSkKCiAgICBiZXN0X2F1YywgYmVzdF9ja3B0ID0gLTEuMCwgTm9uZQogICAgZm9yIGVwb2NoIGluIHJhbmdlKGNmZy5tcGlfZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgZm9yIFhiLCB5YiBpbiB0cl9sbzoKICAgICAgICAgICAgb3B0Lnplcm9fZ3JhZCgpCiAgICAgICAgICAgIGNyaXQobW9kZWwoWGIudG8oZGV2aWNlKSksIHliLnRvKGRldmljZSkpLmJhY2t3YXJkKCkKICAgICAgICAgICAgb3B0LnN0ZXAoKQoKICAgICAgICBtb2RlbC5ldmFsKCk7IHByb2JzLCB5cyA9IFtdLCBbXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBmb3IgWGIsIHliIGluIHZhX2xvOgogICAgICAgICAgICAgICAgcHJvYnMuZXh0ZW5kKAogICAgICAgICAgICAgICAgICAgIHRvcmNoLnNvZnRtYXgobW9kZWwoWGIudG8oZGV2aWNlKSksIDEpWzosIDFdLmNwdSgpLm51bXB5KCkpCiAgICAgICAgICAgICAgICB5cy5leHRlbmQoeWIubnVtcHkoKSkKICAgICAgICB0cnk6ICAgIGF1YyA9IHJvY19hdWNfc2NvcmUoeXMsIHByb2JzKQogICAgICAgIGV4Y2VwdDogYXVjID0gMC41CiAgICAgICAgaWYgYXVjID4gYmVzdF9hdWM6CiAgICAgICAgICAgIGJlc3RfYXVjICA9IGF1YwogICAgICAgICAgICBiZXN0X2NrcHQgPSB7azogdi5jbG9uZSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfQoKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChiZXN0X2NrcHQpOyBtb2RlbC5ldmFsKCkKICAgIHByZWRzLCB5cyA9IFtdLCBbXQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIFhiLCB5YiBpbiB0ZV9sbzoKICAgICAgICAgICAgcHJlZHMuZXh0ZW5kKG1vZGVsKFhiLnRvKGRldmljZSkpLmFyZ21heCgxKS5jcHUoKS5udW1weSgpKQogICAgICAgICAgICB5cy5leHRlbmQoeWIubnVtcHkoKSkKICAgIHJldHVybiBhY2N1cmFjeV9zY29yZSh5cywgcHJlZHMpICogMTAwLjAsIG5wLmFycmF5KHlzKSwgbnAuYXJyYXkocHJlZHMpCgoKZGVmIHJ1bl9vbl9kdW1teSgpOgogICAgZnJvbSB1dGlscy5kdW1teV9kYXRhIGltcG9ydCBsb2FkX21waV9kdW1teQogICAgWCwgeSwgXyA9IGxvYWRfbXBpX2R1bW15KCkKICAgIHByaW50KGYiRHVtbXkgTVBJIHRyYWluaW5nOiBYPXtYLnNoYXBlfSwgZGV2aWNlPXtkZXZpY2V9IikKICAgIGZvciBzZWVkIGluIHJhbmdlKDIpOgogICAgICAgIGFjYywgXywgXyA9IHRyYWluX29uZV9wYWlyKFgsIHksIDAsIDAsIHNlZWQpCiAgICAgICAgcHJpbnQoZiIgIHNlZWQ9e3NlZWR9OiBhY2M9e2FjYzouMmZ9JSIpCiAgICBwcmludCgiRHVtbXkgTVBJIHRyYWluaW5nIHBhc3NlZC4iKQoKCmRlZiBtYWluKHVzZV9kdW1teT1GYWxzZSk6CiAgICBpZiB1c2VfZHVtbXk6CiAgICAgICAgcnVuX29uX2R1bW15KCk7IHJldHVybgoKICAgIG1mICAgID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKFBST0NFU1NFRF9ESVIsICJtYW5pZmVzdC5jc3YiKSkKICAgIHZhbGlkID0gbWZbbWZbInN0YXR1cyJdID09ICJPSyJdCiAgICBwcmludChmIlRyYWluaW5nIG9uIHtsZW4odmFsaWQpfSB2YWxpZCBwYWlycy4gIERldmljZToge2RldmljZX0iKQogICAgY29tcGxldGVkID0gbG9hZF9jb21wbGV0ZWQoKQoKICAgIG91dF9wYXRoID0gb3MucGF0aC5qb2luKFJFU1VMVFNfRElSLCAicmVzdWx0c19tcGkuY3N2IikKICAgIGFsbF9yb3dzID0gKHBkLnJlYWRfY3N2KG91dF9wYXRoKS50b19kaWN0KCJyZWNvcmRzIikKICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG91dF9wYXRoKSBlbHNlIFtdKQoKICAgIGZvciBfLCByb3cgaW4gdmFsaWQuaXRlcnJvd3MoKToKICAgICAgICB1aWQsIGRpZCA9IGludChyb3dbInVzZXJfaWQiXSksIGludChyb3dbImRldmljZV9pZCJdKQogICAgICAgIGRhdGEgICAgID0gbnAubG9hZChvcy5wYXRoLmpvaW4oUFJPQ0VTU0VEX0RJUiwgZiJ7dWlkfV97ZGlkfS5ucHoiKSkKICAgICAgICBYLCB5ICAgICA9IGRhdGFbIlgiXSwgZGF0YVsieSJdCiAgICAgICAgcHJpbnQoZiJcblVzZXI9e3VpZH0gRGV2aWNlPXtkaWR9OiBYPXtYLnNoYXBlfSIpCgogICAgICAgIHNlZWRfYWNjcyA9IFtdCiAgICAgICAgZm9yIHNlZWQgaW4gcmFuZ2UoY2ZnLm5fdHJhaW5pbmdfcnVucyk6CiAgICAgICAgICAgIGlmICh1aWQsIGRpZCwgc2VlZCkgaW4gY29tcGxldGVkOgogICAgICAgICAgICAgICAgcHJpbnQoZiIgIHNlZWQ9e3NlZWR9OiBhbHJlYWR5IGRvbmUg4oCUIHNraXAiKTsgY29udGludWUKICAgICAgICAgICAgYWNjLCBfLCBfID0gdHJhaW5fb25lX3BhaXIoWCwgeSwgdWlkLCBkaWQsIHNlZWQpCiAgICAgICAgICAgIHNlZWRfYWNjcy5hcHBlbmQoYWNjKQogICAgICAgICAgICBwcmludChmIiAgc2VlZD17c2VlZH06IGFjYz17YWNjOi4yZn0lIikKICAgICAgICAgICAgbWFya19jb21wbGV0ZWQodWlkLCBkaWQsIHNlZWQpCgogICAgICAgIGlmIHNlZWRfYWNjczoKICAgICAgICAgICAgYWxsX3Jvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJ1c2VyX2lkIjogICB1aWQsICAgImRldmljZV9pZCI6IGRpZCwKICAgICAgICAgICAgICAgICJtZWFuX2FjYyI6ICByb3VuZChucC5tZWFuKHNlZWRfYWNjcyksIDIpLAogICAgICAgICAgICAgICAgInN0ZF9hY2MiOiAgIHJvdW5kKG5wLnN0ZChzZWVkX2FjY3MpLCAgMiksCiAgICAgICAgICAgICAgICAic3RhdHVzIjogICAgIk9LIn0pCiAgICAgICAgICAgIHBkLkRhdGFGcmFtZShhbGxfcm93cykudG9fY3N2KG91dF9wYXRoLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgcHJpbnQoZiIgIOKGkiB7bnAubWVhbihzZWVkX2FjY3MpOi4xZn0gwrEge25wLnN0ZChzZWVkX2FjY3MpOi4xZn0lIikKCiAgICBmcm9tIGV2YWx1YXRpb24uZXZhbHVhdGUgaW1wb3J0IHByaW50X3RhYmxlMQogICAgcHJpbnRfdGFibGUxKG91dF9wYXRoKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXVzZS1kdW1teSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgbWFpbih1c2VfZHVtbXk9YXJncy51c2VfZHVtbXkpCg==", "training/train_uv.py": "...", "evaluation/__init__.py": "IyBldmFsdWF0aW9uIHBhY2thZ2UK", "evaluation/evaluate.py": "IiIiCkV2YWx1YXRpb24gdXRpbGl0aWVzLgpQYXBlciByZWZlcmVuY2VzOiBTZWN0aW9ucyA0LjIgYW5kIDcgb2YgYXJYaXY6MjMwMi4wMTc1MQoKUnVuOiBweXRob24gZXZhbHVhdGlvbi9ldmFsdWF0ZS5weQpFeHBlY3RlZDogQWxsIGV2YWx1YXRpb24gdW5pdCB0ZXN0cyBwYXNzZWQuCiIiIgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IG9zCgoKZGVmIGNvbXB1dGVfZmFyX2F0X3RhcigKICAgICAgICBnZW51aW5lX3Njb3JlcywKICAgICAgICBpbXBvc3Rvcl9zY29yZXMsCiAgICAgICAgdGFyZ2V0X3RhcjogZmxvYXQgPSAwLjkwLAogICAgICAgIG5fc3RlcHM6IGludCA9IDEwMF8wMDApIC0+IHR1cGxlOgogICAgIiIiCiAgICBNYW51YWwgdGhyZXNob2xkIHN3ZWVwIGZyb20gaGlnaCB0byBsb3cuCiAgICBSZXR1cm5zIChmYXIsIHRocmVzaG9sZCkgYXQgdGhlIGZpcnN0IHBvaW50IHdoZXJlIFRBUiA+PSB0YXJnZXRfdGFyLgogICAgQXZvaWRzIHNrbGVhcm4gaW50ZXJwb2xhdGlvbiBhcnRlZmFjdHMgYXQgdmVyeSBsb3cgRkFSICgxLzUwMDAwKS4KICAgICIiIgogICAgZ2VuID0gbnAuYXNhcnJheShnZW51aW5lX3Njb3JlcywgIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpbXAgPSBucC5hc2FycmF5KGltcG9zdG9yX3Njb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGxlbihnZW4pID09IDAgb3IgbGVuKGltcCkgPT0gMDoKICAgICAgICByZXR1cm4gMS4wLCAwLjAKICAgIGFsbF9zID0gbnAuY29uY2F0ZW5hdGUoW2dlbiwgaW1wXSkKICAgIGZvciB0IGluIG5wLmxpbnNwYWNlKGFsbF9zLm1heCgpLCBhbGxfcy5taW4oKSwgbl9zdGVwcyk6CiAgICAgICAgaWYgbnAubWVhbihnZW4gPj0gdCkgPj0gdGFyZ2V0X3RhcjoKICAgICAgICAgICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oaW1wID49IHQpKSwgZmxvYXQodCkKICAgIHJldHVybiAxLjAsIGZsb2F0KGFsbF9zLm1pbigpKQoKCmRlZiBib290c3RyYXBfZmFyKAogICAgICAgIGdlbnVpbmVfc2NvcmVzLAogICAgICAgIGltcG9zdG9yX3Njb3JlcywKICAgICAgICBuX3JlcGVhdHM6ICBpbnQgICA9IDUwMDAsCiAgICAgICAgdGFyZ2V0X3RhcjogZmxvYXQgPSAwLjkwLAogICAgICAgIHNlZWQ6ICAgICAgIGludCAgID0gMCkgLT4gdHVwbGU6CiAgICAiIiIKICAgIFBhcGVyIFNlY3Rpb24gNi41OiBib290c3RyYXAtbGlrZSBtZXRob2QsIDUwMDAgcmVwZWF0cy4KICAgIENSSVRJQ0FMOiB0aHJlc2hvbGQgaXMgcmUtY2FsaWJyYXRlZCBwZXIgcmVzYW1wbGUgKG5vdCBmaXhlZCkuCiAgICBSZXR1cm5zIChtZWFuX2Zhciwgc3RkX2ZhcikuCiAgICAiIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgZ2VuID0gbnAuYXNhcnJheShnZW51aW5lX3Njb3JlcywgIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpbXAgPSBucC5hc2FycmF5KGltcG9zdG9yX3Njb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGZhcl9saXN0ID0gW10KICAgIGZvciBfIGluIHJhbmdlKG5fcmVwZWF0cyk6CiAgICAgICAgZiwgXyA9IGNvbXB1dGVfZmFyX2F0X3RhcigKICAgICAgICAgICAgcm5nLmNob2ljZShnZW4sIGxlbihnZW4pLCByZXBsYWNlPVRydWUpLAogICAgICAgICAgICBybmcuY2hvaWNlKGltcCwgbGVuKGltcCksIHJlcGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIHRhcmdldF90YXIsCiAgICAgICAgICAgIG5fc3RlcHM9MTBfMDAwKQogICAgICAgIGZhcl9saXN0LmFwcGVuZChmKQogICAgYXJyID0gbnAuYXJyYXkoZmFyX2xpc3QpCiAgICByZXR1cm4gZmxvYXQoYXJyLm1lYW4oKSksIGZsb2F0KGFyci5zdGQoKSkKCgpkZWYgZm9ybWF0X2ZhcihmYXJfdmFsdWU6IGZsb2F0KSAtPiB0dXBsZToKICAgICIiIlJldHVybnMgKGRlY2ltYWxfc3RyLCBmcmFjdGlvbl9zdHIpLiBlLmcuIDAuMDEg4oaSICgnMS4wMGUtMDInLCAnMS8xMDAnKSIiIgogICAgaWYgZmFyX3ZhbHVlID09IDAuMDoKICAgICAgICByZXR1cm4gIjAiLCAiMS9pbmYiCiAgICBrID0gaW50KHJvdW5kKDEuMCAvIGZhcl92YWx1ZSkpCiAgICByZXR1cm4gZiJ7ZmFyX3ZhbHVlOi4yZX0iLCBmIjEve2t9IgoKCmRlZiBydWxlX29mXzMwX2NoZWNrKGZhcl92YWx1ZTogZmxvYXQsIG5faW1wb3N0b3JfY29tcGFyaXNvbnM6IGludCk6CiAgICAiIiIKICAgIFBhcGVyIFNlY3Rpb24gNC4yOiBuZWVkID49IDMwIGVycm9ycyBmb3IgOTAlIGNvbmZpZGVuY2Ugd2l0aGluIMKxMzAlLgogICAgIiIiCiAgICBpZiBmYXJfdmFsdWUgPT0gMC4wOgogICAgICAgIHByaW50KCJSdWxlIG9mIDMwOiBGQVI9MCwgY2Fubm90IGV2YWx1YXRlLiIpOyByZXR1cm4KICAgIG5fZXJyID0gZmFyX3ZhbHVlICogbl9pbXBvc3Rvcl9jb21wYXJpc29ucwogICAgbmVlZCAgPSBpbnQobnAuY2VpbCgzMCAvIGZhcl92YWx1ZSkpCiAgICBzdGF0dXMgPSAiUEFTUyIgaWYgbl9lcnIgPj0gMzAgZWxzZSBmIkZBSUwg4oCUIG5lZWQgPj0ge25lZWR9IGNvbXBhcmlzb25zIgogICAgcHJpbnQoZiJSdWxlIG9mIDMwOiB7bl9lcnI6LjFmfSBlcnJvcnMgLyAiCiAgICAgICAgICBmIntuX2ltcG9zdG9yX2NvbXBhcmlzb25zfSBjb21wYXJpc29ucyDihpIge3N0YXR1c30iKQoKCmRlZiBwcmludF90YWJsZTEoY3N2X3BhdGg6IHN0cik6CiAgICAiIiJQYXBlciBUYWJsZSAxOiBNUEkgYWNjdXJhY3kgcGVyIHVzZXItZGV2aWNlIHBhaXIuIiIiCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoY3N2X3BhdGgpOgogICAgICAgIHByaW50KGYiVGFibGUgMTogbm90IGZvdW5kIOKAlCB7Y3N2X3BhdGh9Iik7IHJldHVybgogICAgZGYgICAgICA9IHBkLnJlYWRfY3N2KGNzdl9wYXRoKQogICAgdXNlcnMgICA9IHNvcnRlZChkZlsidXNlcl9pZCJdLnVuaXF1ZSgpKQogICAgZGV2aWNlcyA9IHNvcnRlZChkZlsiZGV2aWNlX2lkIl0udW5pcXVlKCkpCiAgICBwcmludCgiXG5UYWJsZSAxOiBNUEkgQWNjdXJhY3kgKCUpIikKICAgIHByaW50KGYieydEZXYvVXNlcic6PDEwfSIsIGVuZD0iIikKICAgIGZvciB1IGluIHVzZXJzOgogICAgICAgIHByaW50KGYieydVc2VyJytzdHIodSk6PjE1fSIsIGVuZD0iIikKICAgIHByaW50KCkKICAgIGZvciBkIGluIGRldmljZXM6CiAgICAgICAgcHJpbnQoZiJ7ZDo8MTB9IiwgZW5kPSIiKQogICAgICAgIGZvciB1IGluIHVzZXJzOgogICAgICAgICAgICByID0gZGZbKGRmWyJkZXZpY2VfaWQiXSA9PSBkKSAmIChkZlsidXNlcl9pZCJdID09IHUpXQogICAgICAgICAgICBpZiByLmVtcHR5IG9yIHIuaWxvY1swXVsic3RhdHVzIl0gPT0gIk4vQSI6CiAgICAgICAgICAgICAgICBwcmludChmInsnTi9BJzo+MTV9IiwgZW5kPSIiKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmkgPSByLmlsb2NbMF0KICAgICAgICAgICAgICAgIHZhbCA9IGYie3JpLm1lYW5fYWNjOi4xZn3CsXtyaS5zdGRfYWNjOi4xZn0iCiAgICAgICAgICAgICAgICBwcmludChmInt2YWw6PjE1fSIsIGVuZD0iIikKICAgICAgICBwcmludCgpCgoKZGVmIHByaW50X3RhYmxlMihjc3ZfcGF0aDogc3RyKToKICAgICIiIlBhcGVyIFRhYmxlIDI6IGJhc2VsaW5lIG1vZGVsIG1ldHJpY3MgcGVyIHNwbGl0LiIiIgogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGNzdl9wYXRoKToKICAgICAgICBwcmludChmIlRhYmxlIDI6IG5vdCBmb3VuZCDigJQge2Nzdl9wYXRofSIpOyByZXR1cm4KICAgIGRmID0gcGQucmVhZF9jc3YoY3N2X3BhdGgpCiAgICBwcmludCgiXG5UYWJsZSAyOiBCYXNlbGluZSBQZXJmb3JtYW5jZSIpCiAgICBwcmludChmInsnU3BsaXQnOj42fSB7J0FjY1ZhbCc6PjE0fSB7J0FjY1Rlc3QnOj4xNH0gIgogICAgICAgICAgZiJ7J0ZBUnZhbEA5MCc6PjE4fSB7J0ZBUnRlc3RAOTAnOj4xOH0iKQogICAgZm9yIF8sIHIgaW4gZGYuaXRlcnJvd3MoKToKICAgICAgICBkdiwgZnYgPSBmb3JtYXRfZmFyKHIuZ2V0KCJmYXJfdmFsX21lYW4iLCAgMCkpCiAgICAgICAgZHQsIGZ0ID0gZm9ybWF0X2ZhcihyLmdldCgiZmFyX3Rlc3RfbWVhbiIsIDApKQogICAgICAgIHByaW50KGYie2ludChyWyduX2Jhc2VsaW5lJ10pOj42fSAiCiAgICAgICAgICAgICAgZiJ7clsnYWNjX3ZhbF9tZWFuJ106LjJmfcKxe3JbJ2FjY192YWxfc3RkJ106LjJmfSAgICAgICIKICAgICAgICAgICAgICBmIntyWydhY2NfdGVzdF9tZWFuJ106LjJmfcKxe3JbJ2FjY190ZXN0X3N0ZCddOi4yZn0gICAgICAiCiAgICAgICAgICAgICAgZiJ7ZHZ9KHtmdn0pICAge2R0fSh7ZnR9KSIpCgoKZGVmIHByaW50X3RhYmxlMyhjc3ZfcGF0aDogc3RyKToKICAgICIiIlBhcGVyIFRhYmxlIDM6IHBlci11c2VyIEZBUiBhY3Jvc3MgYWxsIHNwbGl0cy4iIiIKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhjc3ZfcGF0aCk6CiAgICAgICAgcHJpbnQoZiJUYWJsZSAzOiBub3QgZm91bmQg4oCUIHtjc3ZfcGF0aH0iKTsgcmV0dXJuCiAgICBkZiAgICAgPSBwZC5yZWFkX2Nzdihjc3ZfcGF0aCkKICAgIHNwbGl0cyA9IHNvcnRlZChkZlsibl9iYXNlbGluZSJdLnVuaXF1ZSgpKQogICAgcHJpbnQoIlxuVGFibGUgMzogRmluYWwgVGVzdCBGQVIoQFRBUj05MCUpIHBlciB1c2VyIGFuZCBzcGxpdCIpCiAgICBwcmludChmInsndXNlcl9pZCc6Pjh9IiwgZW5kPSIiKQogICAgZm9yIHMgaW4gc3BsaXRzOgogICAgICAgIHByaW50KGYie3N0cihzKTo+MTN9IiwgZW5kPSIiKQogICAgcHJpbnQoKQogICAgZm9yIHVpZCBpbiBzb3J0ZWQoZGZbInVzZXJfaWQiXS51bmlxdWUoKSk6CiAgICAgICAgcHJpbnQoZiJ7dWlkOj44fSIsIGVuZD0iIikKICAgICAgICBmb3IgcyBpbiBzcGxpdHM6CiAgICAgICAgICAgIHIgPSBkZlsoZGZbInVzZXJfaWQiXSA9PSB1aWQpICYgKGRmWyJuX2Jhc2VsaW5lIl0gPT0gcyldCiAgICAgICAgICAgIGlmIHIuZW1wdHk6CiAgICAgICAgICAgICAgICBwcmludChmInsnTi9BJzo+MTN9IiwgZW5kPSIiKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmkgPSByLmlsb2NbMF0KICAgICAgICAgICAgICAgIHZhbCA9IGYie3JpLmZhcl9tZWFuKjEwMDouMWZ9wrF7cmkuZmFyX3N0ZCoxMDA6LjFmfSIKICAgICAgICAgICAgICAgIHByaW50KGYie3ZhbDo+MTN9IiwKICAgICAgICAgICAgICAgICAgICAgIGVuZD0iIikKICAgICAgICBwcmludCgpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgICMgVGVzdCAxOiBGQVIgPSAwCiAgICBnZW4gPSBucC5hcnJheShbMC45LCAwLjg1LCAwLjgsIDAuNzUsIDAuN10pCiAgICBpbXAgPSBucC5hcnJheShbMC4zLCAwLjQsICAwLjUsIDAuMiwgIDAuMV0pCiAgICBmYXIsIHQgPSBjb21wdXRlX2Zhcl9hdF90YXIoZ2VuLCBpbXAsIDAuOTApCiAgICBhc3NlcnQgZmFyID09IDAuMCwgZiJFeHBlY3RlZCAwLjAsIGdvdCB7ZmFyfSIKICAgIHByaW50KGYiVGVzdCAxIOKAlCBGQVI9MDogICAgICAgIEZBUj17ZmFyfSwgdD17dDouNGZ9ICBQQVNTRUQiKQoKICAgICMgVGVzdCAyOiBub24temVybyBGQVIKICAgIGdlbjIgPSBucC5hcnJheShbMC44LCAwLjc1LCAwLjcsIDAuNjUsIDAuNiwgMC41NSwgMC41LCAwLjQ1LCAwLjQsIDAuMzVdKQogICAgaW1wMiA9IG5wLmFycmF5KFswLjc1LCAwLjUsIDAuNCwgMC4zLCAwLjIsIDAuMSwgIDAuMSwgMC4xLCAgMC4xLCAwLjFdKQogICAgZmFyMiwgXyA9IGNvbXB1dGVfZmFyX2F0X3RhcihnZW4yLCBpbXAyLCAwLjkwKQogICAgYXNzZXJ0IDAuMCA8PSBmYXIyIDw9IDEuMAogICAgcHJpbnQoZiJUZXN0IDIg4oCUIG5vbi16ZXJvIEZBUjoge2ZhcjI6LjRmfSAgUEFTU0VEIikKCiAgICAjIFRlc3QgMzogYm9vdHN0cmFwIChyZS1jYWxpYnJhdGVzIHRocmVzaG9sZCBwZXIgcmVzYW1wbGUpCiAgICBtLCBzID0gYm9vdHN0cmFwX2ZhcihnZW4sIGltcCwgbl9yZXBlYXRzPTEwMCwgc2VlZD00MikKICAgIGFzc2VydCBtID49IDAKICAgIHByaW50KGYiVGVzdCAzIOKAlCBib290c3RyYXA6ICAgIG1lYW49e206LjRmfSwgc3RkPXtzOi40Zn0gIFBBU1NFRCIpCgogICAgIyBUZXN0IDQ6IGZvcm1hdF9mYXIKICAgIGZvciB2IGluIFswLjAsIDAuMDEsIDEvNTAwMDBdOgogICAgICAgIGQsIGYgPSBmb3JtYXRfZmFyKHYpCiAgICAgICAgcHJpbnQoZiJUZXN0IDQg4oCUIGZvcm1hdF9mYXIoe3Y6LjZmfSk6IHtkfSwge2Z9IikKCiAgICAjIFRlc3QgNTogcnVsZSBvZiAzMAogICAgcnVsZV9vZl8zMF9jaGVjaygxLzUwMDAwLCAxXzUwMF8wMDApICAgIyBQQVNTICAoMzAgZXJyb3JzIGV4YWN0bHkpCiAgICBydWxlX29mXzMwX2NoZWNrKDEvNTAwMDAsICAgMTAwXzAwMCkgICAjIEZBSUwgICgyIGVycm9ycykKCiAgICBwcmludCgiXG5BbGwgZXZhbHVhdGlvbiB1bml0IHRlc3RzIHBhc3NlZC4iKQo="}

for relpath, b64content in FILES.items():
    fpath = os.path.join(REPO, relpath)
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(base64.b64decode(b64content).decode('utf-8'))

os.chdir(REPO)
if REPO not in os.sys.path:
    os.sys.path.insert(0, REPO)

print(f'Setup complete. {len(FILES)} source files written to {REPO}')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 1: GPU + path + format verification
import torch, os, glob, struct

assert torch.cuda.is_available(), "NO GPU DETECTED!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

BASE = "/kaggle/input/datasets/djaarf"
PATHS = {
    "MPI Part 1": f"{BASE}/motionid-imu-all-motions-part1",
    "MPI Part 2": f"{BASE}/motionid-imu-all-motions-part2",
    "MPI Part 3": f"{BASE}/motionid-imu-all-motions-part3",
    "UV":         f"{BASE}/motionid-imu-specific-motion",
}
all_ok = True
for name, path in PATHS.items():
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'X'} {name}: {path}")
    if not exists: all_ok = False
assert all_ok, "MISSING DATASETS"

sample_accel = glob.glob(f"{BASE}/**/accel.txt", recursive=True)[0]
with open(sample_accel, "rb") as f:
    rec = f.read(20)
ts = struct.unpack(">q", rec[:8])[0]
x, y, z = struct.unpack("<fff", rec[8:20])
print(f"\naccel.txt: {os.path.getsize(sample_accel)/1e6:.1f} MB, {os.path.getsize(sample_accel)//20:,} records")
print(f"First record: ts={ts}, X={x:.6f}, Y={y:.6f}, Z={z:.6f}")

sample_screen = glob.glob(f"{BASE}/**/screen.txt", recursive=True)[0]
with open(sample_screen, "r") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"screen: {line.rstrip()}")

print("\nAll checks passed.")

In [ ]:
# Cell 2: MPI Preprocessing
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from preprocessing.mpi_preprocess import discover_sessions, process_session
from tqdm import tqdm
import pandas as pd, numpy as np

PROCESSED_MPI = "/kaggle/working/data/mpi/processed"
os.makedirs(PROCESSED_MPI, exist_ok=True)

sessions = discover_sessions([
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part1/IMU_all_motions_part1",
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part2/IMU_all_motions_part2",
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part3/IMU_all_motions_part3",
])
print(f"Found {len(sessions)} MPI sessions")

rows = []
for uid, did, sdir in tqdm(sessions, desc="MPI"):
    X, y = process_session(uid, did, sdir)
    key = f"{uid}_{did}"
    if X is None:
        tqdm.write(f"  {uid}/{did}: N/A")
        rows.append({"user_id": uid, "device_id": did, "n_pos": 0, "n_neg": 0, "status": "N/A"})
    else:
        np.savez(os.path.join(PROCESSED_MPI, f"{key}.npz"), X=X, y=y)
        n_pos, n_neg = int((y==1).sum()), int((y==0).sum())
        tqdm.write(f"  {uid}/{did}: X={X.shape} pos={n_pos} neg={n_neg}")
        rows.append({"user_id": uid, "device_id": did, "n_pos": n_pos, "n_neg": n_neg, "status": "OK"})

mf = pd.DataFrame(rows)
mf.to_csv(os.path.join(PROCESSED_MPI, "manifest.csv"), index=False)
print(f"\nMPI done. Valid: {(mf.status=='OK').sum()}/{len(mf)}")

In [ ]:
# Cell 3: UV Preprocessing
import sys, os, numpy as np
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from utils.bin_reader import load_session, FLAG_USER_PRESENT
from preprocessing.uv_preprocess import compute_features, extract_window
from tqdm import tqdm

UV_DIR = "/kaggle/input/datasets/djaarf/motionid-imu-specific-motion/IMU_specific_motion/train_val_test"
PROCESSED_UV = "/kaggle/working/data/uv/processed"
os.makedirs(PROCESSED_UV, exist_ok=True)
assert os.path.exists(UV_DIR), f"Not found: {UV_DIR}"

user_dirs = sorted(d for d in os.listdir(UV_DIR) if os.path.isdir(os.path.join(UV_DIR, d)))
print(f"Found {len(user_dirs)} UV users")

for uid in tqdm(user_dirs, desc="UV"):
    base = os.path.join(UV_DIR, uid, "s20")
    if not os.path.exists(base): continue
    sessions = sorted(os.listdir(base))
    if not sessions: continue
    sdir = os.path.join(base, sessions[0])

    merged, screen = load_session(sdir)
    if merged is None:
        tqdm.write(f"  {uid}: load failed"); continue

    unlocks = screen[screen["event"] == FLAG_USER_PRESENT].sort_values("timestamp")
    feats, clusters, n_skip = [], [], 0
    for trial_idx, (_, row) in enumerate(unlocks.iterrows()):
        w = extract_window(merged, int(row["timestamp"]))
        if w is None: n_skip += 1; continue
        feats.append(compute_features(w))
        clusters.append(trial_idx // 50)

    if not feats: continue
    F = np.stack(feats, 0).astype(np.float32)
    C = np.array(clusters, dtype=np.int64)
    np.savez(os.path.join(PROCESSED_UV, f"{uid}.npz"), features=F, cluster_ids=C, user_id=uid)
    tqdm.write(f"  {uid}: {F.shape}")

done = len([f for f in os.listdir(PROCESSED_UV) if f.endswith(".npz")])
print(f"\\nUV done. {done} users saved.")

In [ ]:
# Cell 4: Shape verification
import os, numpy as np

for label, ddir in [("MPI", "/kaggle/working/data/mpi/processed"),
                     ("UV",  "/kaggle/working/data/uv/processed")]:
    files = [f for f in os.listdir(ddir) if f.endswith(".npz")]
    print(f"=== {label}: {len(files)} files ===")
    if files:
        d = np.load(os.path.join(ddir, files[0]))
        for k in d: print(f"  {k}: {d[k].shape}")
print("\nShape checks passed.")

In [ ]:
# Cell 5: MPI Training
import sys, os, torch
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")
os.makedirs("/kaggle/working/models/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/evaluation", exist_ok=True)

import training.train_mpi as m
m.PROCESSED_DIR = "/kaggle/working/data/mpi/processed"
m.CKPT_DIR      = "/kaggle/working/models/checkpoints"
m.RESULTS_DIR   = "/kaggle/working/evaluation"
m.RESUME_LOG    = "/kaggle/working/evaluation/mpi_completed.csv"

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
m.main(use_dummy=False)

In [ ]:
# Cell 6: UV Training n=75
import sys, os, torch
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

import training.train_uv as m
m.PROCESSED_DIR = "/kaggle/working/data/uv/processed"
m.CKPT_DIR      = "/kaggle/working/models/checkpoints"
m.RESULTS_DIR   = "/kaggle/working/evaluation"

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
m.main(use_dummy=False, n_baseline=75, all_splits=False)

In [ ]:
# Cell 7: UV all 6 splits (optional, ~4 hr)
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

import training.train_uv as m
m.PROCESSED_DIR = "/kaggle/working/data/uv/processed"
m.CKPT_DIR      = "/kaggle/working/models/checkpoints"
m.RESULTS_DIR   = "/kaggle/working/evaluation"

m.main(use_dummy=False, n_baseline=None, all_splits=True)

In [ ]:
# Cell 8: Print Tables
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from evaluation.evaluate import print_table1, print_table2, print_table3

R = "/kaggle/working/evaluation"
print_table1(os.path.join(R, "results_mpi.csv"))
print_table2(os.path.join(R, "results_baseline.csv"))
print_table3(os.path.join(R, "results_uv_final.csv"))

In [ ]:
# Cell 9: List outputs
import os
for dirpath, _, filenames in os.walk("/kaggle/working"):
    for fname in filenames:
        fpath = os.path.join(dirpath, fname)
        rel = os.path.relpath(fpath, "/kaggle/working")
        print(f"  {rel:<60} {os.path.getsize(fpath)/1024:.1f} KB")
print("\nClick Save Version to persist.")